<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_06_temporal_context_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 6: Temporal Context Images and Evidence Metadata

This standalone Colab notebook consumes the **current, persisted Sector 2 → 3 → 4 → 5 artifact chain**. It creates bounded temporal context references, associates official annotation metadata, plans unique video-frame extractions, **verifies** the resulting physical images, and saves the complete Sector 6 bundle. It preserves the existing Sector 4 physical-frame split assignments and every source annotation/bounding box.

The decoder is **NVIDIA NVDEC through PyNvVideoCodec**, sequential from ordinal 0, with 10 independent video workers on GPU 0. Every active video has its own hardware decoder and CUDA stream, with bounded batches of four GPU frames. Only requested frames are copied to host memory for JPEG/PNG encoding.


| Upstream sector | Artifacts used |
|---|---|
| 2 | Completed decode/alignment/video states, confirmed/excluded annotations, audited video inventory, run-matched configuration |
| 3 | Accepted labelled-frame manifest, decoded video/frame manifest, class matrix/support, checked catalog and schema files |
| 4 | Video-level split manifest, labelled members with splits, video roles, adaptation references, checked catalog |
| 5 | Finding-segment manifest, all labelled annotation members with segment IDs, support report, current configuration and checked lineage |

The loader rejects stale or mixed upstream generations rather than selecting a configuration by filename timestamp. It rebases live access paths using committed storage-relative references.


### 1. Dependencies and explicit imports


In [1]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "pyarrow>=16" "opencv-python-headless>=4.10,<6" "PyNvVideoCodec>=2.0.4,<3" torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 74.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from types import CodeType, FunctionType
from concurrent.futures import ThreadPoolExecutor, as_completed
import copy
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import tempfile
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display


### 2. Declarative bootstrap and Sector 6 settings

`None` means: inherit the effective value from the current Sector 5 config; use the declared Sector 6 default only when the upstream key is absent.

Image files and their provenance sidecars remain on mounted persistent storage. GitHub receives the Parquet manifests/results and CSV/JSON reports, schemas, config and catalog under `outputs/phase2/sector6`. Image binaries are not included in Git commits.

In [33]:
BOOTSTRAP = {
    "storage_root": "/content/drive/MyDrive/MMVQA_Clinical",
    "mount_google_drive": True,
    "sector3_dir": None,
    "sector4_dir": None,
    "sector5_dir": None,
    "sector2_results_dir": None,
    "sector2_manifests_dir": None,
    "sector2_configs_dir": None,
}

SECTOR6_SETTINGS = {
    "temporal_context_enabled": None,
    "temporal_context_extract_frames": None,
    "temporal_context_offsets_seconds": None,
    "temporal_context_image_format": None,
    "temporal_context_jpeg_quality": None,
    "temporal_context_png_compression": None,
    "temporal_context_workers": 10,
    "temporal_context_reuse_existing": None,
    "temporal_context_decoder": "nvdec",
    "temporal_context_gpu_id": 0,
    "temporal_context_decode_batch_size": 4,
    "temporal_context_image_dir": None,
    "local_output_dir": "outputs/phase2/sector6",
    "export_dir": "outputs/phase2/sector6",
    "push": True,
    "dry_run": False,
    "github_overrides": {"token_secret_name": "GITHUB_TOKEN"},
}

# Compatibility defaults required by the current Sector 2–4 chain reader.
SECTOR5_DEFAULTS = {"segment_max_gap_frames": 1, "minimum_verified_segment_frames": 2}


### 3. Read-only artifact readers and input contracts

The embedded Sector 5 reader uses the same three Sector 4 inputs as the current Sector 5 notebook: labelled annotations, video/split summaries, and authoritative physical-frame splits. Repeated video keys across different splits are valid; duplicate video/split pairs or conflicting physical-frame assignments are rejected.

These readers preserve the exact catalog hashing/schema conventions of the current Sector 3–5 notebooks. File access occurs only when the explicit `io_*` loading functions are called. Sector 5 members retain all annotation IDs and boxes, including several annotations at one frame/class. Native target annotation collections are checked against their member rows.

In [34]:
# -----------------------------------------------------------------------------------
# Read-only catalog helpers copied from the current Sector 4 reader.
# Their canonical hashes and dtype restoration match the existing Sector 3/4 files.
# -----------------------------------------------------------------------------------

P3_NULL = "\\N"

S3_EXECUTION_METADATA_KEYS = frozenset({
    "run_id", "run_context", "run_datetime", "run_timestamp", "run_date", "run_time",
    "timestamp", "date", "time", "execution_date", "execution_time",
    "created_at", "created_at_utc", "created_utc", "saved_at", "saved_at_utc", "saved_utc",
    "started_at", "started_at_utc", "started_utc", "finished_at", "finished_at_utc", "finished_utc",
    "generated_at", "generated_at_utc", "generated_utc", "updated_at", "updated_at_utc", "updated_utc",
    "modified_at", "modified_at_utc", "modified_utc",
    "source_commit", "source_commit_sha", "source_commit_hash", "source_revision", "config_source_commit",
    "commit", "commit_sha", "head_commit", "current_commit", "repository_commit",
    "publication_receipt", "receipt", "last_publication",
    "source_configs", "config_provenance", "config_conflict_report", "lineage",
    "source_notebooks", "source_config_provenance", "config_sources", "literal_notebooks", "literal_configs",
    "input_sha256", "state_sha256", "upstream_states", "upstream_alignment_run_id",
    "notebook_path", "notebook_sha256", "source_notebook_path", "source_notebook_sha256",
    "source_config_path", "source_config_sha256", "source_snapshot_path", "source_snapshot_sha256",
    "config_snapshot_path", "config_snapshot_sha256", "snapshot_path", "snapshot_sha256",
    "source_config_file", "source_config_hash", "source_snapshot_file", "source_snapshot_hash",
    "config_snapshot_file", "config_snapshot_hash", "snapshot_file", "snapshot_hash",
    "config_inheritance_policy", "taxonomy_validation_policy",
})

def _s3_semantic_json_value(value):
    """Pure conversion to canonical JSON-compatible values; no I/O or clock access."""
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, dict):
        return {str(key): _s3_semantic_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_s3_semantic_json_value(item) for item in value]
    if isinstance(value, (set, frozenset)):
        values = [_s3_semantic_json_value(item) for item in value]
        return sorted(values, key=lambda item: json.dumps(item, sort_keys=True))
    if isinstance(value, (Path, PurePosixPath)):
        return str(value)
    if isinstance(value, np.generic):
        return _s3_semantic_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        raise ValueError("Semantic configuration must not contain NaN or infinite values.")
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError(f"Unsupported semantic configuration value: {type(value).__name__}")

def _s3_semantic_json(value):
    return json.dumps(_s3_semantic_json_value(value), ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def _s3_semantic_digest(value):
    return hashlib.sha256(_s3_semantic_json(value).encode("utf-8")).hexdigest()

def sector3_semantic_config(config):
    """Project authored effective values, excluding execution/source bookkeeping only.

    Full source snapshots and current execution metadata remain available in the
    saved envelope. Authored paths, thresholds, publication settings (including
    push/dry_run), and unknown substantive config keys retain their meaning.
    """
    if not isinstance(config, dict):
        raise TypeError("Sector 3 semantic configuration requires a dictionary.")
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S3_EXECUTION_METADATA_KEYS}
        if isinstance(value, (list, tuple)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)

def sector3_semantic_config_fingerprint(config):
    """Stable identity of authored configuration, independent of execution time."""
    return _s3_semantic_digest(sector3_semantic_config(config))

def _p3_json_default(value):
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, pd.DataFrame):
        return {"type": "dataframe", "columns": list(value.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in value.dtypes.items()},
                "records": value.astype(object).where(value.notna(), None).to_dict("records")}
    if isinstance(value, (Path, PurePosixPath, datetime, pd.Timestamp)):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, tuple)):
        return sorted(value) if isinstance(value, set) else list(value)
    raise TypeError(f"Unsupported Sector 3 JSON value: {type(value).__name__}")

def _p3_json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                      allow_nan=False, default=_p3_json_default) + "\n"

def _p3_json_hash(value):
    return hashlib.sha256(_p3_json_text(value).encode("utf-8")).hexdigest()

def _p3_file_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

def _p3_relative(value):
    path = PurePosixPath(str(value))
    if path.is_absolute() or ".." in path.parts or len(path.parts) != 2:
        raise ValueError(f"Sector 3 artifacts must be flat relative paths: {value!r}")
    if path.parts[0] not in {"configs", "results", "reports"}:
        raise ValueError(f"Unexpected Sector 3 artifact group: {value!r}")
    return path.as_posix()

def _p3_safe_path(root, relative):
    path = root / _p3_relative(relative)
    if path.is_symlink() or root not in path.resolve().parents or path.parent.is_symlink():
        raise RuntimeError(f"Unsafe Sector 3 artifact path: {relative}")
    return path

def _p3_schema(frame):
    columns = []
    for column, dtype in frame.dtypes.items():
        descriptor = {"name": column, "dtype": str(dtype)}
        if isinstance(dtype, pd.StringDtype):
            descriptor["string_storage"] = dtype.storage
        if str(dtype) == "object":
            descriptor["object_kind"] = pd.api.types.infer_dtype(frame[column], skipna=True)
        if isinstance(dtype, pd.CategoricalDtype):
            descriptor.update({"categories": list(dtype.categories), "ordered": dtype.ordered})
        columns.append(descriptor)
    return {"rows": len(frame), "columns": columns, "column_axis_name": frame.columns.name,
            "csv_null_marker": P3_NULL, "schema_version": 1}

def sector3_table_fingerprint(frame):
    """Hash every value and its dtype schema; no clock or execution metadata is added."""
    if frame.columns.duplicated().any() or not all(isinstance(column, str) for column in frame.columns):
        raise ValueError("Sector 3 tables require unique string column names.")
    normalized = frame.reset_index(drop=True)
    digest = hashlib.sha256(_p3_json_text(_p3_schema(normalized)).encode("utf-8"))
    try:
        # Missing values have one semantic identity, including None/NaN/pd.NA in object columns.
        hash_values = normalized.astype(object).where(normalized.notna(), None)
        values = pd.util.hash_pandas_object(hash_values, index=False, categorize=False)
    except TypeError as error:
        raise TypeError("Sector 3 tables must contain scalar values; canonicalize nested conflict values first.") from error
    digest.update(values.to_numpy(dtype="uint64").astype("<u8", copy=False).tobytes())
    return digest.hexdigest()

def _p3_restore_csv(path, schema):
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    expected = [descriptor["name"] for descriptor in schema["columns"]]
    if list(frame.columns) != expected or len(frame) != schema["rows"]:
        raise RuntimeError(f"CSV dimensions differ from saved schema: {path.name}")
    for descriptor in schema["columns"]:
        name, dtype = descriptor["name"], descriptor["dtype"]
        values = frame[name]
        nulls = values.eq(P3_NULL)
        values = values.mask(nulls, pd.NA)
        if dtype == "object":
            kind = descriptor.get("object_kind", "string")
            if kind in {"integer", "floating", "mixed-integer-float", "decimal"}:
                values = pd.to_numeric(values, errors="raise").astype(object)
            elif kind == "boolean":
                values = values.map({"True": True, "False": False}).astype(object)
            elif kind not in {"string", "unicode", "bytes", "empty"}:
                raise RuntimeError(f"Unsupported object CSV dtype {kind!r}: canonicalize this table before saving.")
            else:
                values = values.astype(object)
            frame[name] = values.mask(nulls, None)
        elif dtype in {"bool", "boolean"}:
            if (~nulls & ~values.isin(["True", "False"])).any():
                raise RuntimeError(f"Invalid saved boolean values: {path.name}/{name}")
            frame[name] = values.map({"True": True, "False": False}).astype(dtype)
        elif dtype == "string":
            frame[name] = values.astype(pd.StringDtype(storage=descriptor.get("string_storage", "python")))
        elif dtype == "category":
            category = pd.CategoricalDtype(descriptor["categories"], ordered=descriptor["ordered"])
            if pd.api.types.infer_dtype(descriptor["categories"], skipna=True) in {"integer", "floating", "mixed-integer-float"}:
                values = pd.to_numeric(values, errors="raise")
            frame[name] = values.astype(category)
        elif dtype.startswith("timedelta64"):
            frame[name] = pd.to_timedelta(values).astype(dtype)
        else:
            frame[name] = values.astype(dtype)
    frame.columns.name = schema.get("column_axis_name")
    return frame

def _p3_read_table(root, descriptor):
    path = _p3_safe_path(root, descriptor["relative_path"])
    schema_path = _p3_safe_path(root, descriptor["schema_path"])
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    if schema.get("fingerprint") != descriptor["fingerprint"] or schema.get("artifact") != descriptor["relative_path"]:
        raise RuntimeError("Table schema disagrees with the committed catalog.")
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else _p3_restore_csv(path, schema)
    if sector3_table_fingerprint(frame) != descriptor["fingerprint"]:
        raise RuntimeError(f"Typed table readback fingerprint failed: {path.name}")
    return frame

def _p3_catalog_meaning(state):
    return {key: state[key] for key in (
        "schema_version", "complete", "processing_signature", "semantic_config_fingerprint",
        "input_sha256", "core_source_sha256", "config_snapshot", "tables", "summary", "artifacts", "obsolete_paths")}

S4_EXECUTION_METADATA_KEYS = frozenset({
    'run_id', 'run_context', 'run_datetime', 'run_timestamp', 'run_date', 'run_time',
    'timestamp', 'date', 'time', 'execution_date', 'execution_time',
    'created_at', 'created_at_utc', 'created_utc', 'saved_at', 'saved_at_utc', 'saved_utc',
    'started_at', 'started_at_utc', 'started_utc', 'finished_at', 'finished_at_utc', 'finished_utc',
    'generated_at', 'generated_at_utc', 'generated_utc', 'updated_at', 'updated_at_utc', 'updated_utc',
    'modified_at', 'modified_at_utc', 'modified_utc',
})

def sector4_semantic_config(config):
    """Exclude clock bookkeeping only from the effective authored Sector 4 config."""
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S4_EXECUTION_METADATA_KEYS}
        if isinstance(value, (tuple, list)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)

def sector4_semantic_config_fingerprint(config):
    return _s3_semantic_digest(sector4_semantic_config(config))


S5_SECTOR3_INPUTS = (
    "labelled_frame_manifest", "video_frame_manifest",
    "video_class_matrix", "video_class_support",
)
S5_SECTOR4_RESULTS = (
    "labelled_frame_manifest_with_split",
    "video_split_manifest",
    "frame_split_manifest",
)
S5_REQUIRED_LABELLED_COLUMNS = {
    "annotation_id", "annotation_status", "alignment_training_eligible",
    "video_key", "finding_class", "finding_class_normalized",
    "clinical_group", "frame_number", "opencv_frame_index", "frame_index_offset",
    "filename", "image_path", "image_relpath", "has_bbox", "bbox_spatially_usable",
    "bbox_validation_status", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax", "split",
}


def _s5_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return value


def _s5_checked_hash(path, expected):
    if not isinstance(expected, str) or re.fullmatch(r"[0-9a-f]{64}", expected) is None:
        raise ValueError(f"Missing or invalid committed SHA-256: {path}")
    actual = _p3_file_hash(path)
    if actual != expected:
        raise RuntimeError(f"Artifact changed after its completed audit: {path}")
    return actual


def io_s5_read_catalog(root, sector):
    """Read one current Sector 3/4 catalog, never choose a config by timestamp."""
    root = Path(root).resolve()
    state_path = _p3_safe_path(root, f"configs/sector{sector}_state.json")
    if not state_path.is_file():
        raise FileNotFoundError(f"Run/save Sector {sector} first; current catalog missing: {state_path}")
    state_hash = _p3_file_hash(state_path)
    state = _s5_json(state_path)
    if state.get("schema_version") != 2 or state.get("complete") is not True:
        raise RuntimeError(f"Sector {sector} catalog is incomplete or has an unsupported schema.")
    if state.get("content_id") != _p3_json_hash(_p3_catalog_meaning(state)):
        raise RuntimeError(f"Sector {sector} catalog content fingerprint is invalid.")
    records = {item["relative_path"]: item for item in state["artifacts"]}
    operational = {f"configs/sector{sector}_state.json", f"configs/sector{sector}_publication_receipt.json"}
    if len(records) != len(state["artifacts"]) or set(records) & operational:
        raise RuntimeError(f"Sector {sector} catalog has duplicate or operational artifact entries.")
    for relative, record in records.items():
        path = _p3_safe_path(root, relative)
        if not path.is_file() or path.stat().st_size != record["bytes"]:
            raise RuntimeError(f"Sector {sector} cataloged file missing or size changed: {relative}")
        _s5_checked_hash(path, record["sha256"])
    if state["config_snapshot"] not in records:
        raise RuntimeError(f"Sector {sector} current config snapshot is absent from its catalog.")
    snapshot_path = _p3_safe_path(root, state["config_snapshot"])
    envelope = _s5_json(snapshot_path)
    semantic = sector3_semantic_config_fingerprint if sector == 3 else sector4_semantic_config_fingerprint
    if (envelope.get("semantic_config_fingerprint") != state["semantic_config_fingerprint"]
            or semantic(envelope["effective_config"]) != state["semantic_config_fingerprint"]):
        raise RuntimeError(f"Sector {sector} effective config disagrees with its catalog.")
    for name, descriptor in state["tables"].items():
        if not {descriptor["relative_path"], descriptor["schema_path"]}.issubset(records):
            raise RuntimeError(f"Sector {sector} table/schema absent from catalog: {name}")
        schema = _s5_json(_p3_safe_path(root, descriptor["schema_path"]))
        if schema.get("artifact") != descriptor["relative_path"] or schema.get("fingerprint") != descriptor["fingerprint"]:
            raise RuntimeError(f"Sector {sector} table schema disagrees with its catalog: {name}")
    summary = state["summary"]
    if not {summary["summary_relative_path"], summary["report_relative_path"]}.issubset(records):
        raise RuntimeError(f"Sector {sector} summary/report absent from catalog.")
    if (_p3_json_hash(_s5_json(_p3_safe_path(root, summary["summary_relative_path"]))) != summary["fingerprint"]
            or records[summary["report_relative_path"]]["sha256"] != summary["report_fingerprint"]):
        raise RuntimeError(f"Sector {sector} summary/report disagrees with its catalog.")
    for relative in state.get("obsolete_paths", []):
        if not _p3_relative(relative).startswith("results/") or relative in records:
            raise RuntimeError(f"Sector {sector} catalog has an invalid obsolete-result entry.")
    return {"root": root, "state": state, "config": envelope["effective_config"],
            "state_path": state_path, "state_sha256": state_hash,
            "snapshot_path": snapshot_path, "snapshot_sha256": _p3_file_hash(snapshot_path)}


def _s5_required_tables(context, names):
    missing = sorted(set(names) - set(context["state"]["tables"]))
    if missing:
        raise KeyError(f"Current catalog at {context['root']} lacks required tables: {missing}")
    return {name: _p3_read_table(context["root"], context["state"]["tables"][name]) for name in names}


def _s5_rebase_declared_directory(value, old_storage_root, storage_root):
    path = Path(value).expanduser()
    if not path.is_absolute():
        return (storage_root / path).resolve()
    try:
        relative = path.resolve().relative_to(Path(old_storage_root).expanduser().resolve())
    except ValueError:
        if Path(old_storage_root).expanduser().resolve() == storage_root:
            return path.resolve()
        raise ValueError("Cannot rebase an upstream absolute directory outside its storage_root; use a BOOTSTRAP directory override.") from None
    return (storage_root / relative).resolve()


def _s5_sector2_directories(source_config, bootstrap, storage_root):
    def declared(value):
        return _s5_rebase_declared_directory(value, source_config["storage_root"], storage_root)
    output = declared(source_config["output_dir"])
    defaults = {
        "results": declared(source_config["results_dir"]) if "results_dir" in source_config else output / "results",
        "manifests": declared(source_config["curated_data_dir"]) / "manifests",
        "configs": output / "configs",
    }
    return {name: Path(bootstrap[f"sector2_{name}_dir"]).expanduser().resolve()
            if bootstrap.get(f"sector2_{name}_dir") else default.resolve()
            for name, default in defaults.items()}


def _s5_completed_state(path):
    state = _s5_json(path)
    if state.get("complete") is not True:
        raise RuntimeError(f"Sector 2 processing is incomplete: {path}")
    return state


def _s5_identity_multiset(table, columns):
    values = table[columns].astype("string")
    for column in columns:
        if column in {"frame_number", "opencv_frame_index", "aligned_frame_index"} or column.startswith("bbox_"):
            values[column] = pd.to_numeric(table[column], errors="coerce").astype("Float64").astype("string")
    return values.fillna("").value_counts(sort=False).sort_index()


def _s5_validate_frame_splits(labelled, split_manifest, frame_split_manifest):
    """Validate authoritative physical-frame assignments and video/split memberships."""
    frame_keys = ["video_key", "opencv_frame_index"]
    pair_keys = ["video_key", "split"]
    allowed = ["train", "validation", "test"]
    for name, table, keys in (
        ("labelled annotations", labelled, frame_keys),
        ("frame_split_manifest", frame_split_manifest, frame_keys),
        ("video_split_manifest", split_manifest, pair_keys),
    ):
        missing = sorted(set(keys + ["split"]) - set(table.columns))
        if missing:
            raise KeyError(f"Sector 4 {name} lacks split fields: {missing}")
        for column in keys + ["split"]:
            values = table[column].astype("string")
            if (values.isna() | values.str.strip().eq("")).any():
                raise ValueError(f"Sector 4 {name} has null/empty {column} values.")
        if not table["split"].isin(allowed).all():
            raise ValueError(f"Sector 4 {name} contains unsupported split names.")
    if frame_split_manifest.duplicated(frame_keys).any():
        raise ValueError("Sector 4 frame_split_manifest requires one row per video/decoded-frame key.")
    if split_manifest.duplicated(pair_keys).any():
        raise ValueError("Sector 4 video_split_manifest requires unique (video_key, split) pairs.")
    indices = frame_split_manifest["opencv_frame_index"]
    numeric = pd.to_numeric(indices, errors="coerce")
    if (numeric.isna() | numeric.lt(0) | numeric.mod(1).ne(0)).any() or (
        not np.isfinite(numeric.astype("float64")).all()
        or indices.map(lambda value: isinstance(value, (bool, np.bool_))).any()
    ):
        raise ValueError("Sector 4 frame_split_manifest requires finite non-negative integer decoded-frame indices.")
    frame_assignments = labelled[frame_keys + ["split"]].drop_duplicates()
    if frame_assignments.duplicated(frame_keys).any():
        raise ValueError("All annotations/classes on one physical frame must share the same split.")
    reference = frame_split_manifest[frame_keys + ["split"]].copy()
    reference["opencv_frame_index"] = numeric.astype("Int64")
    comparison = frame_assignments.merge(
        reference, on=frame_keys, how="outer", validate="one_to_one",
        suffixes=("_annotation", "_manifest"), indicator=True,
    )
    if not comparison["_merge"].eq("both").all() or not comparison[
        "split_annotation"
    ].eq(comparison["split_manifest"]).fillna(False).all():
        raise ValueError("Accepted annotations disagree with the authoritative Sector 4 frame split manifest (frame coverage or split assignment).")
    membership = frame_assignments[pair_keys].drop_duplicates().merge(
        split_manifest[pair_keys], on=pair_keys, how="outer",
        validate="one_to_one", indicator=True,
    )
    if not membership["_merge"].eq("both").all():
        raise ValueError("Sector 4 video/split memberships disagree with the accepted frame assignments.")
    expected_counts = labelled.groupby(pair_keys, observed=True, sort=True).agg(
        retained_annotation_count=("annotation_id", "size"),
        retained_unique_frame_count=("opencv_frame_index", "nunique"),
        retained_class_count=("finding_class_normalized", "nunique"),
    )
    saved_counts = split_manifest.set_index(pair_keys).reindex(expected_counts.index)
    for column in expected_counts:
        if column in saved_counts and not pd.to_numeric(
            saved_counts[column], errors="coerce"
        ).eq(expected_counts[column]).fillna(False).all():
            raise ValueError(f"Sector 4 video/split summary has inconsistent {column} values.")


def _s5_validate_loaded_frames(labelled, sector3_labelled, videos, split_manifest, confirmed, frame_split_manifest):
    missing = sorted(S5_REQUIRED_LABELLED_COLUMNS - set(labelled))
    if missing:
        raise KeyError(f"Sector 4 labelled-frame artifact lacks Sector 5 input fields: {missing}")
    if labelled.empty:
        raise ValueError("Sector 5 requires at least one accepted labelled annotation.")
    for column in ("annotation_id", "video_key", "finding_class_normalized", "clinical_group", "split"):
        values = labelled[column].astype("string")
        if (values.isna() | values.str.strip().eq("")).any():
            raise ValueError(f"Sector 5 input has missing/empty {column} values.")
    if not labelled["annotation_id"].is_unique:
        raise ValueError("Sector 4 must preserve unique source annotation_id values.")
    for column in ("frame_number", "opencv_frame_index", "frame_index_offset"):
        values = pd.to_numeric(labelled[column], errors="coerce")
        invalid = values.isna() | ~np.isfinite(values.astype("float64")) | values.mod(1).ne(0)
        if column != "frame_index_offset":
            invalid = invalid | values.lt(0)
        if invalid.any() or labelled[column].map(lambda value: isinstance(value, (bool, np.bool_))).any():
            raise ValueError(f"Sector 5 input requires genuine finite integer {column} values.")
        labelled[column] = values.astype("Int64")
    if not labelled["opencv_frame_index"].eq(labelled["frame_number"] + labelled["frame_index_offset"]).all():
        raise ValueError("Accepted decoded indices disagree with source frame numbers plus verified offsets.")
    if labelled.groupby("video_key", observed=True)["frame_index_offset"].nunique().gt(1).any():
        raise ValueError("Accepted annotations have inconsistent verified offsets within a video.")
    if not labelled["annotation_status"].eq("source_expert_annotation_alignment_confirmed").all():
        raise ValueError("Sector 5 can consume only expert annotations with confirmed alignment.")
    if not labelled["alignment_training_eligible"].eq(True).all():
        raise ValueError("Sector 5 input includes annotations rejected by alignment.")
    if videos["video_key"].isna().any() or not videos["video_key"].is_unique:
        raise ValueError("The completed decoded video inventory needs unique non-null video keys.")
    bounds = labelled["video_key"].map(videos.set_index("video_key")["frame_count"])
    if bounds.isna().any() or not labelled["opencv_frame_index"].lt(bounds).all():
        raise ValueError("Accepted decoded frame indices lie outside audited video bounds.")
    if not videos["frame_count_source"].eq("completed_sector2_decode_audit").all():
        raise ValueError("Video bounds must come from completed Sector 2 decoding.")
    _s5_validate_frame_splits(labelled, split_manifest, frame_split_manifest)
    # Preserve every expert bbox row. Compare annotation identities and multiplicities,
    # and never turn duplicate frame/class rows into new labels or discard them here.
    identity_columns = [column for column in (
        "annotation_id", "filename", "image_key", "video_key", "frame_number",
        "opencv_frame_index", "finding_class_normalized", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax",
    ) if column in sector3_labelled]
    if not _s5_identity_multiset(labelled, identity_columns).equals(_s5_identity_multiset(sector3_labelled, identity_columns)):
        raise ValueError("Sector 4 labelled annotations differ from the current Sector 3 evidence.")
    source_columns = [column for column in (
        "filename", "image_key", "video_key", "frame_number", "finding_class_normalized",
        "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax",
    ) if column in confirmed and column in sector3_labelled]
    if not _s5_identity_multiset(confirmed, source_columns).equals(_s5_identity_multiset(sector3_labelled, source_columns)):
        raise ValueError("Sector 3 accepted annotations do not preserve current Sector 2 confirmed evidence.")


def _s5_rebase_live_paths(table, storage_root):
    result = table.copy()
    for path_column, relative_column in (("image_path", "image_relpath"), ("video_path", "video_relpath")):
        if path_column not in result or relative_column not in result:
            continue
        paths = []
        for value in result[relative_column]:
            if pd.isna(value):
                paths.append(pd.NA)
                continue
            relative = PurePosixPath(str(value))
            if relative.is_absolute() or "\\" in str(value) or any(part in {"", ".", ".."} for part in str(value).split("/")):
                raise ValueError(f"Unsafe portable path in {relative_column}: {value!r}")
            paths.append(str(storage_root.joinpath(*relative.parts)))
        result[path_column] = pd.Series(paths, index=result.index).astype(result[path_column].dtype)
    return result


def io_load_sector5_context(bootstrap, settings):
    """Read/verify the current 2 -> 3 -> 4 chain; return config, inputs and lineage."""
    if not isinstance(bootstrap, dict) or not isinstance(settings, dict):
        raise TypeError("BOOTSTRAP and SECTOR5_SETTINGS must be dictionaries.")
    allowed_settings = set(SECTOR5_DEFAULTS) | {"local_output_dir", "export_dir", "push", "dry_run", "github_overrides"}
    unknown = sorted(set(settings) - allowed_settings)
    if unknown:
        raise KeyError(f"Unexpected Sector 5 algorithm settings: {unknown}")
    storage_root = Path(bootstrap["storage_root"]).expanduser().resolve()
    root4 = Path(bootstrap["sector4_dir"]).expanduser().resolve() if bootstrap.get("sector4_dir") else storage_root / "outputs/phase2/sector4"
    context4 = io_s5_read_catalog(root4, 4)
    source4 = context4["config"]
    declared3 = source4.get("sector3", {}).get("local_output_dir", "outputs/phase2/sector3")
    root3 = Path(bootstrap["sector3_dir"]).expanduser().resolve() if bootstrap.get("sector3_dir") else _s5_rebase_declared_directory(declared3, source4["storage_root"], storage_root)
    context3 = io_s5_read_catalog(root3, 3)
    inputs3 = _s5_required_tables(context3, S5_SECTOR3_INPUTS)
    inputs4 = _s5_required_tables(context4, S5_SECTOR4_RESULTS)
    source2 = context3["config"].get("source_configs", {}).get("sector2")
    if not isinstance(source2, dict):
        raise KeyError("Current Sector 3 config snapshot lacks its Sector 2 source configuration.")
    dirs2 = _s5_sector2_directories(source2, bootstrap, storage_root)
    state_paths2 = {"alignment": dirs2["results"] / "frame_alignment_state.json",
                    "decode": dirs2["results"] / "decode_audit_state.json",
                    "manifest": dirs2["manifests"] / "video_manifest_state.json"}
    states2 = {name: _s5_completed_state(path) for name, path in state_paths2.items()}
    state_hashes2 = {name: _p3_file_hash(path) for name, path in state_paths2.items()}
    run_id = states2["alignment"].get("run_id")
    source2_path = None
    if run_id:
        matches = [(path, _s5_json(path)) for path in dirs2["configs"].glob("phase2_02_video_frame_validation_*_config.json")]
        matches = [(path, config) for path, config in matches if config.get("run_id") == run_id]
        if not matches:
            raise FileNotFoundError(f"Sector 2 alignment run {run_id!r} requires its saved config snapshot in {dirs2['configs']}.")
        source2_path, source2 = sorted(matches, key=lambda pair: str(pair[0]))[0]
        if any(config != source2 for _, config in matches):
            raise ValueError("Multiple Sector 2 snapshots share the alignment run_id with different configurations.")
        if _s5_sector2_directories(source2, bootstrap, storage_root) != dirs2:
            raise ValueError("Run-matched Sector 2 configuration selects different artifact directories.")
    threshold_keys = {"min_ssim": "frame_alignment_min_ssim", "min_margin": "frame_alignment_min_margin",
                      "candidate_offsets": "frame_index_offset_candidates"}
    if any(states2["alignment"].get("effective_settings", {}).get(recorded) != source2.get(key)
           for recorded, key in threshold_keys.items()):
        raise ValueError("Sector 2 config snapshot disagrees with the completed alignment decision settings.")
    hashes2 = {}
    for family, required in (("alignment", {"frame_alignment_confirmed.parquet", "frame_alignment_excluded.csv"}),
                             ("decode", {"decode_audit.csv"})):
        committed = states2[family].get("output_sha256", {})
        if not required.issubset(committed):
            raise ValueError(f"Sector 2 {family} state lacks required committed output hashes.")
        for filename, digest in committed.items():
            if Path(filename).name != filename or "/" in filename or "\\" in filename:
                raise ValueError("Sector 2 state must list simple committed output filenames.")
            hashes2[filename] = _s5_checked_hash(dirs2["results"] / filename, digest)
    hashes2["video_manifest.csv"] = _s5_checked_hash(dirs2["manifests"] / "video_manifest.csv", states2["manifest"].get("csv_sha256"))
    prior2 = context3["state"]["input_sha256"]
    for filename in ("frame_alignment_confirmed.parquet", "decode_audit.csv", "video_manifest.csv"):
        recorded = {digest for path, digest in prior2.items() if PurePosixPath(str(path)).name == filename}
        if recorded != {hashes2[filename]}:
            raise RuntimeError(f"Current Sector 3 was not built from current Sector 2 {filename}; rerun/save Sector 3 then Sector 4.")
    for name in S5_SECTOR3_INPUTS:
        descriptor = context3["state"]["tables"][name]
        for field in ("relative_path", "schema_path"):
            actual = _p3_file_hash(_p3_safe_path(root3.resolve(), descriptor[field]))
            if context4["state"]["input_sha256"].get(f"{name}/{field}") != actual:
                raise RuntimeError(f"Current Sector 4 was not built from current Sector 3 {name}/{field}; rerun/save Sector 4.")
    raw2 = {"df_alignment_confirmed": pd.read_parquet(dirs2["results"] / "frame_alignment_confirmed.parquet"),
            "frame_alignment_excluded": pd.read_csv(dirs2["results"] / "frame_alignment_excluded.csv", dtype="string"),
            "decode_audit": pd.read_csv(dirs2["results"] / "decode_audit.csv", dtype={"video_key": "string"}),
            "video_manifest": pd.read_csv(dirs2["manifests"] / "video_manifest.csv", dtype={"video_key": "string"})}
    if (len(raw2["df_alignment_confirmed"]) != states2["alignment"].get("confirmed_annotation_rows")
            or len(raw2["frame_alignment_excluded"]) != states2["alignment"].get("excluded_annotation_rows")
            or len(raw2["df_alignment_confirmed"]) + len(raw2["frame_alignment_excluded"]) != states2["alignment"].get("original_annotation_rows")):
        raise ValueError("Sector 2 confirmed/excluded annotation counts disagree with its completed alignment state.")
    if states2["decode"].get("all_videos_decoded") is not True:
        raise ValueError("Every Sector 2 source video must finish decoding before Sector 5.")
    for name, family in (("video_manifest", "manifest"), ("decode_audit", "decode")):
        table = raw2[name]
        if (len(table) != states2[family].get("video_count") or table["video_key"].isna().any()
                or not table["video_key"].is_unique):
            raise ValueError(f"Sector 2 {name} identities/counts disagree with its completed state.")
    if set(raw2["video_manifest"]["video_key"]) != set(raw2["decode_audit"]["video_key"]):
        raise ValueError("Sector 2 video inventory and decode audit contain different video keys.")
    labelled = inputs4["labelled_frame_manifest_with_split"].copy()
    _s5_validate_loaded_frames(labelled, inputs3["labelled_frame_manifest"], inputs3["video_frame_manifest"],
                              inputs4["video_split_manifest"], raw2["df_alignment_confirmed"],
                              inputs4["frame_split_manifest"])
    config = copy.deepcopy(source4)
    decisions = []
    for key, fallback in SECTOR5_DEFAULTS.items():
        override = settings.get(key)
        value = override if override is not None else source4.get(key, fallback)
        minimum = 1 if key == "segment_max_gap_frames" else 2
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
            raise ValueError(f"{key} must be an integer >= {minimum}.")
        config[key] = int(value)
        decisions.append({"config_key": key, "effective_value": int(value),
                          "resolution": "sector5_override" if override is not None else "inherited_sector4" if key in source4 else "declared_sector5_default"})
    sector5_settings = copy.deepcopy(settings)
    sector5_settings.setdefault("local_output_dir", "outputs/phase2/sector5")
    sector5_settings.setdefault("export_dir", "outputs/phase2/sector5")
    if sector5_settings["export_dir"] != "outputs/phase2/sector5":
        raise ValueError("Sector 5 export_dir must be outputs/phase2/sector5.")
    for flag in ("push", "dry_run"):
        if flag in settings and not isinstance(settings[flag], bool):
            raise ValueError(f"SECTOR5_SETTINGS[{flag!r}] must be a boolean.")
    github = copy.deepcopy(source4.get("github", {}))
    github_overrides = copy.deepcopy(settings.get("github_overrides", {}))
    if not isinstance(github_overrides, dict) or set(github_overrides) - {"token_secret_name"}:
        raise ValueError("Sector 5 github_overrides supports only token_secret_name; repository and branch are inherited.")
    if "token_secret_name" in github_overrides:
        name = github_overrides["token_secret_name"]
        if not isinstance(name, str) or not name.strip() or re.search(r"\s", name):
            raise ValueError("token_secret_name must be a nonempty secret reference without whitespace.")
    github.update(github_overrides)
    github.update({"repo_output_dir": sector5_settings["export_dir"],
                   "push": settings.get("push", github.get("push", False)),
                   "dry_run": settings.get("dry_run", github.get("dry_run", False)),
                   "commit_message": "Phase 2 Sector 5: publish finding segments and temporal supervision"})
    github.pop("confirm_push", None)
    config.update({"phase": 2, "sector": 5, "storage_root": str(storage_root),
                   "sector5": sector5_settings, "github": github})
    for key in S4_EXECUTION_METADATA_KEYS:
        config.pop(key, None)
    # Copies used for access are rebased only after validating the saved bytes and identities.
    runtime3 = {name: _s5_rebase_live_paths(table, storage_root) for name, table in inputs3.items()}
    runtime4 = {name: _s5_rebase_live_paths(table, storage_root) for name, table in inputs4.items()}
    runtime4["labelled_frame_manifest_with_split"] = _s5_rebase_live_paths(labelled, storage_root)
    inputs = {"sector2": raw2, "sector3": runtime3, "sector4": runtime4}
    consumed = {f"sector2/{name}": digest for name, digest in hashes2.items()}
    consumed_paths = {f"sector2/{name}": str((dirs2["manifests"] if name == "video_manifest.csv" else dirs2["results"]) / name)
                      for name in hashes2}
    for name, path in state_paths2.items():
        consumed[f"sector2/state/{name}"] = state_hashes2[name]
        consumed_paths[f"sector2/state/{name}"] = str(path)
    if source2_path is not None:
        consumed["sector2/config_snapshot"] = _p3_file_hash(source2_path)
        consumed_paths["sector2/config_snapshot"] = str(source2_path)
    for sector, context, names in ((3, context3, S5_SECTOR3_INPUTS), (4, context4, S5_SECTOR4_RESULTS)):
        for name in names:
            for field in ("relative_path", "schema_path"):
                relative = context["state"]["tables"][name][field]
                consumed[f"sector{sector}/{name}/{field}"] = _p3_file_hash(_p3_safe_path(context["root"], relative))
                consumed_paths[f"sector{sector}/{name}/{field}"] = str(_p3_safe_path(context["root"], relative))
        consumed[f"sector{sector}/state"] = context["state_sha256"]
        consumed_paths[f"sector{sector}/state"] = str(context["state_path"])
        consumed[f"sector{sector}/config_snapshot"] = context["snapshot_sha256"]
        consumed_paths[f"sector{sector}/config_snapshot"] = str(context["snapshot_path"])
    lineage = {"input_sha256": consumed, "input_descriptors": consumed_paths, "sector2_run_id": run_id,
               "sector2_states": states2, "sector2_state_sha256": state_hashes2,
               "sector2_config_snapshot": str(source2_path) if source2_path else None,
               "sector2_config_snapshot_sha256": _p3_file_hash(source2_path) if source2_path else None,
               "upstream": {f"sector{sector}": {"root": str(context["root"]), "content_id": context["state"]["content_id"],
                   "state_sha256": context["state_sha256"], "config_snapshot": str(context["snapshot_path"]),
                   "config_snapshot_sha256": context["snapshot_sha256"]} for sector, context in ((3, context3), (4, context4))},
               "config_inheritance_report": pd.DataFrame.from_records(decisions),
               "config_inheritance_policy": "Current verified Sector 4 effective config; explicit Sector 5 overrides; declared defaults only when absent."}
    for context in (context3, context4):
        _s5_checked_hash(context["state_path"], context["state_sha256"])
    for name, path in state_paths2.items():
        _s5_checked_hash(path, state_hashes2[name])
    for filename, expected in hashes2.items():
        parent = dirs2["manifests"] if filename == "video_manifest.csv" else dirs2["results"]
        _s5_checked_hash(parent / filename, expected)
    return {"config": config, "inputs": inputs, "lineage": lineage}


In [35]:
# -----------------------------------------------------------------------------------
# Sector 6 checked input loading and declarative configuration.
# Requires the read-only Sector 5 catalog/helper block; no cell execution or I/O
# occurs until io_load_sector6_context is called.
# -----------------------------------------------------------------------------------


SECTOR6_DEFAULTS = {
    "temporal_context_enabled": True,
    "temporal_context_extract_frames": True,
    "temporal_context_offsets_seconds": [-2.0, -1.0, 0.0, 1.0, 2.0],
    "temporal_context_image_format": "jpg",
    "temporal_context_jpeg_quality": 95,
    "temporal_context_png_compression": 3,
    "temporal_context_workers": 10,
    "temporal_context_reuse_existing": True,
    "temporal_context_decoder": "nvdec",
    "temporal_context_gpu_id": 0,
    "temporal_context_decode_batch_size": 4,
    "temporal_context_image_dir": "curated/phase2/temporal_frames",
}
S6_SECTOR5_REQUIRED_TABLES = (
    "finding_segment_manifest", "labelled_frame_manifest_with_segments",
)


def _s6_portable_path(value, storage_root, field):
    """Rebase a declared portable path without touching the filesystem."""
    if value is None or value is pd.NA or pd.isna(value):
        return pd.NA
    text = str(value)
    relative = PurePosixPath(text)
    if (not text or relative.is_absolute() or "\\" in text
            or any(part in {"", ".", ".."} for part in text.split("/"))):
        raise ValueError(f"Unsafe storage-relative path in {field}: {value!r}")
    return str(Path(storage_root).joinpath(*relative.parts))


def _s6_rebase_live_paths(table, storage_root):
    """Return a copy with live paths rebuilt only from committed relpaths."""
    result = _s5_rebase_live_paths(table, storage_root)
    for path_column, relative_column in (
        ("target_image_path", "target_image_relpath"),
        ("context_image_path", "context_image_relpath"),
    ):
        if path_column in result and relative_column in result:
            paths = result[relative_column].map(
                lambda value: _s6_portable_path(value, storage_root, relative_column))
            result[path_column] = paths.astype(result[path_column].dtype)
    if "target_annotations_json" in result:
        def rebase_annotations(value):
            if pd.isna(value):
                return pd.NA
            annotations = json.loads(str(value))
            if not isinstance(annotations, list) or any(not isinstance(item, dict) for item in annotations):
                raise ValueError("Sector 5 target_annotations_json must contain an annotation list.")
            for annotation in annotations:
                if "image_path" in annotation:
                    if "image_relpath" not in annotation:
                        raise KeyError("A saved target annotation lacks its portable image_relpath.")
                    path = _s6_portable_path(annotation["image_relpath"], storage_root, "target_annotations_json/image_relpath")
                    annotation["image_path"] = None if pd.isna(path) else path
            return json.dumps(annotations, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
        result["target_annotations_json"] = result["target_annotations_json"].map(rebase_annotations).astype("string")
    return result


def _s6_integer_setting(value, name, minimum, maximum=None):
    if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer))
            or value < minimum or (maximum is not None and value > maximum)):
        interval = f"{minimum}..{maximum}" if maximum is not None else f">= {minimum}"
        raise ValueError(f"{name} must be an integer {interval}.")
    return int(value)


def _s6_validate_effective_settings(config):
    """Pure validation/normalization of the effective Sector 6 settings."""
    result = copy.deepcopy(config)
    for name in ("temporal_context_enabled", "temporal_context_extract_frames", "temporal_context_reuse_existing"):
        if not isinstance(result[name], bool):
            raise ValueError(f"{name} must be a boolean.")
    offsets = result["temporal_context_offsets_seconds"]
    if not isinstance(offsets, (list, tuple)) or not offsets:
        raise ValueError("temporal_context_offsets_seconds must be a nonempty numeric list containing zero.")
    if any(isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.integer, np.floating)) for value in offsets):
        raise ValueError("Temporal offsets must contain genuine numbers, not booleans or strings.")
    offsets = [float(value) for value in offsets]
    if not np.isfinite(offsets).all() or len(set(offsets)) != len(offsets) or 0.0 not in offsets:
        raise ValueError("Temporal offsets must be finite, unique and include 0.0 for the verified target.")
    result["temporal_context_offsets_seconds"] = sorted(offsets)
    extension = str(result["temporal_context_image_format"]).strip().lower().lstrip(".")
    if extension not in {"jpg", "jpeg", "png"}:
        raise ValueError("temporal_context_image_format must be jpg, jpeg or png.")
    result["temporal_context_image_format"] = extension
    result["temporal_context_jpeg_quality"] = _s6_integer_setting(result["temporal_context_jpeg_quality"], "temporal_context_jpeg_quality", 1, 100)
    result["temporal_context_png_compression"] = _s6_integer_setting(result["temporal_context_png_compression"], "temporal_context_png_compression", 0, 9)
    result["temporal_context_workers"] = _s6_integer_setting(result["temporal_context_workers"], "temporal_context_workers", 1)
    result["temporal_context_gpu_id"] = _s6_integer_setting(result["temporal_context_gpu_id"], "temporal_context_gpu_id", 0)
    result["temporal_context_decode_batch_size"] = _s6_integer_setting(result["temporal_context_decode_batch_size"], "temporal_context_decode_batch_size", 1)
    if result["temporal_context_decoder"] != "nvdec":
        raise ValueError("Sector 6 supports only temporal_context_decoder='nvdec'; CPU fallback is disabled.")
    image_dir = result["temporal_context_image_dir"]
    if not isinstance(image_dir, str):
        raise ValueError("temporal_context_image_dir must be a storage-relative directory string.")
    _s6_portable_path(image_dir, result["storage_root"], "temporal_context_image_dir")
    return result


def _s6_validate_sector5_tables(inputs5, current4, video_frames):
    """Preserve all source expert annotations and current physical-frame split assignments."""
    labelled = inputs5["labelled_frame_manifest_with_segments"]
    segments = inputs5["finding_segment_manifest"]
    required_labelled = set(current4.columns) | {"finding_segment_id"}
    missing = sorted(required_labelled - set(labelled.columns))
    if missing:
        raise KeyError(f"Sector 5 member manifest lacks preserved Sector 4 fields: {missing}")
    if (labelled["annotation_id"].isna().any() or not labelled["annotation_id"].is_unique
            or set(labelled["annotation_id"]) != set(current4["annotation_id"])):
        raise ValueError("Sector 5 annotation IDs do not preserve the current accepted Sector 4 evidence.")
    left = labelled.loc[:, current4.columns].sort_values("annotation_id", kind="stable").reset_index(drop=True)
    right = current4.sort_values("annotation_id", kind="stable").reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(left, right, check_dtype=False, check_categorical=False)
    except AssertionError as error:
        raise ValueError("Sector 5 changed accepted source annotations, bounding boxes or Sector 4 splits.") from error
    required_segments = {
        "finding_segment_id", "video_key", "finding_class", "finding_class_normalized", "clinical_group", "split",
        "verified_start_frame", "verified_end_frame", "verified_frame_count", "verified_annotation_count",
        "target_frame_number", "target_opencv_frame_index", "target_image_path", "target_image_relpath",
        "target_image_annotation_id", "target_annotation_ids_json", "target_annotations_json",
        "target_annotation_count", "temporal_grounding_candidate", "verified_temporal_boundary_available", "temporal_supervision_level",
    }
    missing = sorted(required_segments - set(segments.columns))
    if missing:
        raise KeyError(f"Sector 5 segment manifest lacks Sector 6 fields: {missing}")
    if segments["finding_segment_id"].isna().any() or not segments["finding_segment_id"].is_unique:
        raise ValueError("Sector 5 segment IDs must be unique and non-null.")
    if labelled["finding_segment_id"].isna().any() or set(labelled["finding_segment_id"]) != set(segments["finding_segment_id"]):
        raise ValueError("Sector 5 segment identities disagree with annotation memberships.")
    if not segments["temporal_grounding_candidate"].isin([True, False]).all():
        raise ValueError("Sector 5 temporal-grounding flags must be non-null booleans.")
    if not segments["verified_temporal_boundary_available"].eq(False).all():
        raise ValueError("Observed finding anchors cannot be treated as verified clinical boundaries.")
    group = labelled.groupby("finding_segment_id", sort=False, observed=True)
    invariants = ["video_key", "finding_class_normalized", "clinical_group", "split"]
    if group[invariants].nunique(dropna=False).gt(1).any(axis=1).any():
        raise ValueError("One finding segment contains conflicting source video, finding, taxonomy or split.")
    expected = group.agg(
        video_key=("video_key", "first"), finding_class_normalized=("finding_class_normalized", "first"),
        clinical_group=("clinical_group", "first"), split=("split", "first"),
        verified_start_frame=("frame_number", "min"), verified_end_frame=("frame_number", "max"),
        verified_frame_count=("frame_number", "nunique"), verified_annotation_count=("annotation_id", "size"),
    ).sort_index()
    actual = segments.set_index("finding_segment_id").loc[expected.index, expected.columns]
    try:
        # Identical validated IDs may use object or StringDtype after Parquet
        # readback/groupby. Ignore index storage dtype only; labels, order,
        # index names and every summary value still have to agree.
        pd.testing.assert_frame_equal(actual, expected, check_dtype=False,
                                      check_index_type=False, check_categorical=False)
    except AssertionError as error:
        raise ValueError("Sector 5 summary counts, observed ranges or split assignments disagree with every retained annotation.") from error
    lookup = labelled.set_index("annotation_id")
    anchor_positions = labelled.groupby(["finding_segment_id", "frame_number"], sort=False, observed=True).indices
    for target in segments.to_dict("records"):
        segment_id = target["finding_segment_id"]
        positions = anchor_positions.get((segment_id, target["target_frame_number"]), np.empty(0, dtype="int64"))
        members = labelled.iloc[positions]
        ids = json.loads(str(target["target_annotation_ids_json"]))
        annotation_records = json.loads(str(target["target_annotations_json"]))
        if (members.empty or not isinstance(ids, list) or len(ids) != len(set(ids))
                or set(ids) != set(members["annotation_id"])
                or int(target["target_annotation_count"]) != len(members)
                or not isinstance(annotation_records, list)
                or any(not isinstance(item, dict) for item in annotation_records)
                or {item.get("annotation_id") for item in annotation_records} != set(ids)
                or len(annotation_records) != len(members)):
            raise ValueError(f"Sector 5 target annotation collection is incomplete: {segment_id}")
        if target["target_image_annotation_id"] not in ids:
            raise ValueError(f"Sector 5 representative annotation does not belong to its target frame: {segment_id}")
        representative = lookup.loc[target["target_image_annotation_id"]]
        for target_column, member_column in (
            ("video_key", "video_key"), ("finding_class_normalized", "finding_class_normalized"),
            ("split", "split"), ("target_frame_number", "frame_number"),
            ("target_opencv_frame_index", "opencv_frame_index"),
            ("target_image_relpath", "image_relpath"), ("target_image_path", "image_path"),
        ):
            saved, source = target[target_column], representative[member_column]
            if pd.isna(saved) or pd.isna(source) or saved != source:
                raise ValueError(f"Sector 5 representative target disagrees with its source annotation: {segment_id}/{target_column}")
        for annotation in annotation_records:
            source = lookup.loc[annotation["annotation_id"]]
            for name, saved in annotation.items():
                if name == "annotation_id":
                    continue
                if name not in source.index:
                    raise ValueError(f"Sector 5 target annotation has an unknown source field: {name}")
                live = source[name]
                saved_missing, live_missing = pd.isna(saved), pd.isna(live)
                if saved_missing != live_missing or (not saved_missing and saved != live):
                    raise ValueError(f"Sector 5 target annotation collection changed expert metadata: {segment_id}/{name}")
    bounds = segments["video_key"].map(video_frames.set_index("video_key")["frame_count"])
    if bounds.isna().any() or not segments["target_opencv_frame_index"].ge(0).all() or not segments["target_opencv_frame_index"].lt(bounds).all():
        raise ValueError("Sector 5 target decoded indices lie outside current completed Sector 2 video bounds.")


def io_load_sector6_context(bootstrap, settings):
    """Read and verify the current 2 -> 3 -> 4 -> 5 artifact chain.

    Every upstream byte hash is checked before input construction and again
    before returning. No artifact is selected by timestamp or notebook globals.
    """
    if not isinstance(bootstrap, dict) or not isinstance(settings, dict):
        raise TypeError("BOOTSTRAP and SECTOR6_SETTINGS must be dictionaries.")
    allowed = set(SECTOR6_DEFAULTS) | {"local_output_dir", "export_dir", "push", "dry_run", "github_overrides"}
    unknown = sorted(set(settings) - allowed)
    if unknown:
        raise KeyError(f"Unexpected Sector 6 settings: {unknown}")
    storage_root = Path(bootstrap["storage_root"]).expanduser().resolve()
    root5 = Path(bootstrap["sector5_dir"]).expanduser().resolve() if bootstrap.get("sector5_dir") else storage_root / "outputs/phase2/sector5"
    context5 = io_s5_read_catalog(root5, 5)
    source5 = context5["config"]
    if source5.get("phase") != 2 or source5.get("sector") != 5:
        raise ValueError("The current Sector 5 catalog must contain a Phase 2 Sector 5 effective configuration.")
    inherited_s5_settings = {key: source5.get(key) for key in SECTOR5_DEFAULTS}
    inherited_s5_settings.update({key: source5.get("sector5", {}).get(key, default) for key, default in (
        ("local_output_dir", "outputs/phase2/sector5"), ("export_dir", "outputs/phase2/sector5"),
    )})
    inherited_s5_settings.update({"push": source5.get("github", {}).get("push", False),
                                  "dry_run": source5.get("github", {}).get("dry_run", False)})
    context234 = io_load_sector5_context(bootstrap, inherited_s5_settings)
    current_hashes = context234["lineage"]["input_sha256"]
    if context5["state"].get("input_sha256") != current_hashes:
        prior = context5["state"].get("input_sha256", {})
        changed = sorted(key for key in set(prior) | set(current_hashes) if prior.get(key) != current_hashes.get(key))
        raise RuntimeError("Current Sector 5 was not built from the current completed Sector 2/3/4 chain. "
                           f"Rerun/save Sector 5 first. Changed lineage entries: {changed}")
    missing = sorted(set(S6_SECTOR5_REQUIRED_TABLES) - set(context5["state"]["tables"]))
    if missing:
        raise KeyError(f"Current Sector 5 catalog lacks required tables: {missing}")
    inputs5 = _s5_required_tables(context5, tuple(context5["state"]["tables"]))
    runtime5 = {name: _s6_rebase_live_paths(table, storage_root) for name, table in inputs5.items()}
    _s6_validate_sector5_tables(runtime5,
        context234["inputs"]["sector4"]["labelled_frame_manifest_with_split"],
        context234["inputs"]["sector3"]["video_frame_manifest"])
    config = copy.deepcopy(source5)
    config["storage_root"] = str(storage_root)
    decisions = []
    for key, fallback in SECTOR6_DEFAULTS.items():
        override = settings.get(key)
        value = override if override is not None else source5.get(key, fallback)
        config[key] = copy.deepcopy(value)
        decisions.append({"config_key": key, "effective_value_json": _s3_semantic_json(value),
                          "resolution": "sector6_override" if override is not None else
                          "inherited_sector5" if key in source5 else "declared_sector6_default"})
    config = _s6_validate_effective_settings(config)
    sector6 = copy.deepcopy(settings)
    sector6.setdefault("local_output_dir", "outputs/phase2/sector6")
    sector6.setdefault("export_dir", "outputs/phase2/sector6")
    if sector6["export_dir"] != "outputs/phase2/sector6":
        raise ValueError("Sector 6 export_dir must be outputs/phase2/sector6.")
    local_dir = sector6["local_output_dir"]
    if not isinstance(local_dir, str) or not local_dir.strip():
        raise ValueError("Sector 6 local_output_dir must be a nonempty directory string.")
    if not Path(local_dir).expanduser().is_absolute():
        _s6_portable_path(local_dir, storage_root, "local_output_dir")
    github = copy.deepcopy(source5.get("github", {}))
    overrides = copy.deepcopy(settings.get("github_overrides", {}))
    if not isinstance(overrides, dict) or set(overrides) - {"token_secret_name"}:
        raise ValueError("Sector 6 github_overrides supports only token_secret_name; repository and branch are inherited.")
    if "token_secret_name" in overrides:
        secret_name = overrides["token_secret_name"]
        if not isinstance(secret_name, str) or not secret_name.strip() or re.search(r"\s", secret_name):
            raise ValueError("token_secret_name must be a nonempty secret reference without whitespace.")
    for flag in ("push", "dry_run"):
        if flag in settings and not isinstance(settings[flag], bool):
            raise ValueError(f"SECTOR6_SETTINGS[{flag!r}] must be a boolean.")
    github.update(overrides)
    github.update({"repo_output_dir": sector6["export_dir"],
                   "push": settings.get("push", github.get("push", False)),
                   "dry_run": settings.get("dry_run", github.get("dry_run", False)),
                   "commit_message": "Phase 2 Sector 6: publish temporal evidence and extraction audit"})
    github.pop("confirm_push", None)
    config.update({"phase": 2, "sector": 6, "sector6": sector6, "github": github})
    for key in S4_EXECUTION_METADATA_KEYS:
        config.pop(key, None)
    lineage = copy.deepcopy(context234["lineage"])
    hashes = lineage["input_sha256"]
    descriptors = lineage["input_descriptors"]
    for name, descriptor in context5["state"]["tables"].items():
        for field in ("relative_path", "schema_path"):
            path = _p3_safe_path(context5["root"], descriptor[field])
            hashes[f"sector5/{name}/{field}"] = _p3_file_hash(path)
            descriptors[f"sector5/{name}/{field}"] = str(path)
    hashes["sector5/state"] = context5["state_sha256"]
    descriptors["sector5/state"] = str(context5["state_path"])
    hashes["sector5/config_snapshot"] = context5["snapshot_sha256"]
    descriptors["sector5/config_snapshot"] = str(context5["snapshot_path"])
    lineage["sector5_state"] = copy.deepcopy(context5["state"])
    lineage["sector5_state_sha256"] = context5["state_sha256"]
    lineage["upstream"]["sector5"] = {
        "root": str(context5["root"]), "content_id": context5["state"]["content_id"],
        "state_sha256": context5["state_sha256"], "config_snapshot": str(context5["snapshot_path"]),
        "config_snapshot_sha256": context5["snapshot_sha256"],
    }
    lineage["config_inheritance_report"] = pd.DataFrame.from_records(decisions)
    lineage["config_inheritance_policy"] = (
        "Current verified Sector 5 effective config; explicit Sector 6 overrides; declared defaults only when absent.")
    for name, expected in hashes.items():
        _s5_checked_hash(descriptors[name], expected)
    inputs = copy.copy(context234["inputs"])
    inputs["sector5"] = runtime5
    return {"config": config, "inputs": inputs, "lineage": lineage}


### 4. Mount storage and load the checked Sector 2–5 chain

This is the only upstream input-loading step. Relative paths are preferred for portability; optional bootstrap overrides can locate existing artifact directories.

In [36]:
if BOOTSTRAP.get("mount_google_drive", False) and str(BOOTSTRAP["storage_root"]).startswith("/content/drive/"):
    try:
        from google.colab import drive
    except ImportError:
        if not Path(BOOTSTRAP["storage_root"]).is_dir():
            raise RuntimeError("Open this notebook in Colab, or set storage_root to a mounted local copy.") from None
    else:
        drive.mount("/content/drive", force_remount=False)

SECTOR6_CONTEXT = io_load_sector6_context(BOOTSTRAP, SECTOR6_SETTINGS)
CONFIG = SECTOR6_CONTEXT["config"]
INPUTS = SECTOR6_CONTEXT["inputs"]
LINEAGE = SECTOR6_CONTEXT["lineage"]
SECTOR2_INPUTS, SECTOR3_INPUTS = INPUTS["sector2"], INPUTS["sector3"]
SECTOR4_INPUTS, SECTOR5_INPUTS = INPUTS["sector4"], INPUTS["sector5"]

labelled_frame_manifest = SECTOR5_INPUTS["labelled_frame_manifest_with_segments"]
finding_segment_manifest = SECTOR5_INPUTS["finding_segment_manifest"]
# Audited bounds, FPS and source paths are already verified in the Sector 3 inventory.
video_manifest = SECTOR3_INPUTS["video_frame_manifest"]
sector6_root = Path(CONFIG["storage_root"]) / CONFIG["sector6"]["local_output_dir"]
DIRS = {
    "sector6_dir": str(sector6_root),
    "reports_dir": str(sector6_root / "reports"),
    "temporal_frames_dir": str(Path(CONFIG["storage_root"]) / CONFIG["temporal_context_image_dir"]),
}

display(pd.DataFrame([
    {"sector": sector, "table": name, "rows": len(table)}
    for sector, tables in INPUTS.items() for name, table in tables.items()
]))
display(LINEAGE["config_inheritance_report"])
print("PASS: current Sector 2–5 artifacts, annotations, splits and lineage checked.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,sector,table,rows
0,sector2,df_alignment_confirmed,47239
1,sector2,frame_alignment_excluded,0
2,sector2,decode_audit,117
3,sector2,video_manifest,117
4,sector3,labelled_frame_manifest,47239
5,sector3,video_frame_manifest,117
6,sector3,video_class_matrix,43
7,sector3,video_class_support,117
8,sector4,labelled_frame_manifest_with_split,47239
9,sector4,video_manifest_with_roles,117


,config_key,effective_value_json,resolution
0,temporal_context_enabled,true,declared_sector6_default
1,temporal_context_extract_frames,true,declared_sector6_default
2,temporal_context_offsets_seconds,"[-2.0,-1.0,0.0,1.0,2.0]",declared_sector6_default
3,temporal_context_image_format,"""jpg""",declared_sector6_default
4,temporal_context_jpeg_quality,95,declared_sector6_default
5,temporal_context_png_compression,3,declared_sector6_default
6,temporal_context_workers,10,sector6_override
7,temporal_context_reuse_existing,true,declared_sector6_default
8,temporal_context_decoder,"""nvdec""",sector6_override
9,temporal_context_gpu_id,0,sector6_override


PASS: current Sector 2–5 artifacts, annotations, splits and lineage checked.


### 5. Pure, data-oriented transformations

These functions read no files, write no files and do not mutate their input DataFrames. Joins have explicit key cardinality. Extraction planning deduplicates **physical video/frame requests**, while temporal references remain one row per segment/offset. Annotation enrichment derives separate frame and frame/class lookup tables without removing source annotations.

Each eligible segment retains its verified target; out-of-bounds context is clipped and reported as an incomplete window. Two offsets that round to the same frame are rejected. Official images always take priority, including frames annotated with another finding.

In [37]:
# -----------------------------------------------------------------------------------
# Sector 6 pure transformations. Every input/output is explicit; no file access.
# -----------------------------------------------------------------------------------

S6_TEMPORAL_COLUMNS = [
    'finding_segment_id', 'video_key', 'video_relpath', 'finding_class',
    'finding_class_normalized', 'split', 'target_frame_number', 'target_opencv_frame_index',
    'target_image_annotation_id', 'target_image_path', 'target_image_relpath',
    'temporal_supervision_level', 'verified_temporal_boundary_available',
    'temporal_offset_seconds', 'effective_temporal_offset_seconds', 'frame_delta',
    'context_frame_number', 'opencv_frame_index', 'frame_index_offset', 'is_target',
    'requested_context_count', 'available_context_count', 'temporal_window_complete',
]
S6_WINDOW_COLUMNS = [
    'finding_segment_id', 'video_key', 'split', 'finding_class_normalized',
    'requested_context_count', 'available_context_count', 'temporal_window_complete',
]
TEMPORAL_EXTRACTION_REQUEST_COLUMNS = [
    'video_key', 'opencv_frame_index', 'frame_index_offset', 'video_path', 'video_relpath',
    'output_path', 'output_relpath',
]
TEMPORAL_EXTRACTION_RESULT_COLUMNS = [
    'video_key', 'opencv_frame_index', 'extracted_image_path', 'extracted_image_relpath',
    'extraction_status', 'extraction_error', 'source_video_sha256',
    'extracted_image_sha256', 'decoder_backend',
]
TEMPORAL_EXTRACTION_FAILURE_COLUMNS = [
    'finding_segment_id', 'video_key', 'opencv_frame_index', 'temporal_offset_seconds',
    'context_image_path', 'context_image_source', 'context_annotation_status',
    'extraction_status', 'extraction_error', 'image_validation_error',
]
S6_AUDIT_COLUMNS = [
    'context_image_path', 'context_image_relpath', 'image_readable', 'image_sha256',
    'image_width', 'image_height', 'image_validation_error',
]
S6_INT_COLUMNS = frozenset({
    'target_frame_number', 'target_opencv_frame_index', 'frame_delta', 'context_frame_number',
    'opencv_frame_index', 'frame_index_offset', 'requested_context_count',
    'available_context_count', 'context_verified_annotation_count', 'image_width',
    'image_height', 'row_count', 'context_official_image_count', 'context_image_annotation_count',
})
S6_BOOL_COLUMNS = frozenset({
    'is_target', 'temporal_window_complete', 'verified_temporal_boundary_available',
    'context_is_verified', 'context_is_verified_same_finding', 'image_readable',
})
S6_FLOAT_COLUMNS = frozenset({'temporal_offset_seconds', 'effective_temporal_offset_seconds'})


def s6_typed_table(frame):
    """Normalize scalar table dtypes, including empty tables, without mutating inputs."""
    result = frame.copy()
    for column in result:
        dtype = ('Int64' if column in S6_INT_COLUMNS else
                 'boolean' if column in S6_BOOL_COLUMNS else
                 'Float64' if column in S6_FLOAT_COLUMNS else 'string')
        result[column] = result[column].astype(dtype)
    return result.reset_index(drop=True)


def s6_empty_table(columns):
    return s6_typed_table(pd.DataFrame(columns=columns))


def s6_require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame))
    if missing:
        raise KeyError(f'{name} lacks columns: {missing}')


def s6_integer_series(values, name, minimum=None):
    numeric = pd.to_numeric(values, errors='coerce')
    invalid = (numeric.isna() | ~np.isfinite(numeric.astype('float64')) |
               numeric.mod(1).ne(0) | values.map(lambda x: isinstance(x, (bool, np.bool_))))
    if minimum is not None:
        invalid |= numeric.lt(minimum)
    if invalid.any():
        raise ValueError(f'{name} requires finite integers' + (f' >= {minimum}' if minimum is not None else '') + '.')
    return numeric.astype('Int64')


def build_verified_video_offsets(labelled_manifest):
    """Keep a verified mapping per video; never infer a new global frame offset."""
    s6_require_columns(labelled_manifest, ['video_key', 'frame_number', 'opencv_frame_index',
                                         'frame_index_offset'], 'Labelled manifest')
    offsets = labelled_manifest[['video_key', 'frame_index_offset']].drop_duplicates().copy()
    offsets['frame_index_offset'] = s6_integer_series(offsets['frame_index_offset'], 'Verified offsets')
    if offsets['video_key'].isna().any() or not offsets['video_key'].is_unique:
        raise ValueError('Every labelled video needs one non-null verified offset.')
    source_numbers = s6_integer_series(labelled_manifest['frame_number'], 'Labelled source frame numbers', 0)
    decoder_indices = s6_integer_series(labelled_manifest['opencv_frame_index'], 'Labelled decoder indices', 0)
    annotation_offsets = s6_integer_series(labelled_manifest['frame_index_offset'], 'Verified annotation offsets')
    if not decoder_indices.eq(source_numbers + annotation_offsets).all():
        raise ValueError('Labelled source numbers disagree with confirmed decoder indices.')
    return s6_typed_table(offsets.sort_values('video_key', kind='stable'))


def build_temporal_manifest(segment_manifest, video_manifest, config, frame_index_offsets):
    """Build bounded evidence references using audited counts and per-video offsets.

    Seconds use container FPS and rounded decoder-frame deltas. They are nominal
    offsets, not a VFR timestamp claim or verified clinical onset/offset.
    """
    s6_require_columns(segment_manifest, [
        'finding_segment_id', 'video_key', 'finding_class', 'finding_class_normalized',
        'split', 'target_frame_number', 'target_opencv_frame_index',
        'target_image_annotation_id', 'target_image_path', 'target_image_relpath',
        'temporal_grounding_candidate', 'temporal_supervision_level',
        'verified_temporal_boundary_available'], 'Segment manifest')
    s6_require_columns(video_manifest, ['video_key', 'video_relpath', 'frame_count',
                                       'container_fps'], 'Audited video manifest')
    s6_require_columns(frame_index_offsets, ['video_key', 'frame_index_offset'], 'Verified offsets')
    if not video_manifest['video_key'].is_unique or not frame_index_offsets['video_key'].is_unique:
        raise ValueError('Video metadata and verified offsets must have unique video keys.')
    if not segment_manifest['finding_segment_id'].is_unique:
        raise ValueError('Finding-segment IDs must be unique.')
    flags = segment_manifest['temporal_grounding_candidate'].astype('boolean')
    if flags.isna().any():
        raise ValueError('Temporal-grounding candidate flags cannot be missing.')
    if not config['temporal_context_enabled']:
        return s6_empty_table(S6_TEMPORAL_COLUMNS)
    candidates = segment_manifest.loc[flags].copy()
    if candidates.empty:
        return s6_empty_table(S6_TEMPORAL_COLUMNS)
    raw_offsets = config['temporal_context_offsets_seconds']
    if any(isinstance(x, (bool, np.bool_)) for x in raw_offsets):
        raise ValueError('Temporal offsets must be real numbers, not booleans.')
    offsets = pd.to_numeric(pd.Series(raw_offsets, dtype='object'), errors='coerce')
    if (offsets.empty or offsets.isna().any() or not np.isfinite(offsets.astype('float64')).all()
            or offsets.duplicated().any() or not offsets.eq(0).any()):
        raise ValueError('Temporal offsets must be unique finite numbers including 0.')
    metadata = video_manifest[['video_key', 'video_relpath', 'frame_count', 'container_fps']]
    combined = candidates.merge(metadata, on='video_key', how='left', validate='many_to_one')
    combined = combined.merge(frame_index_offsets, on='video_key', how='left', validate='many_to_one')
    combined['frame_count'] = s6_integer_series(combined['frame_count'], 'Audited frame counts', 1)
    combined['frame_index_offset'] = s6_integer_series(combined['frame_index_offset'], 'Verified offsets')
    for column in ('target_frame_number', 'target_opencv_frame_index'):
        combined[column] = s6_integer_series(combined[column], column, 0)
    fps = pd.to_numeric(combined['container_fps'], errors='coerce')
    if (fps.isna() | ~np.isfinite(fps.astype('float64')) | fps.le(0)).any():
        raise ValueError('Every candidate video requires a positive finite container FPS.')
    combined['container_fps'] = fps.astype('float64')
    if (not combined['target_opencv_frame_index'].eq(
            combined['target_frame_number'] + combined['frame_index_offset']).all()
            or not combined['target_opencv_frame_index'].lt(combined['frame_count']).all()):
        raise ValueError('Segment targets disagree with confirmed offsets or audited video bounds.')
    expanded = combined.merge(pd.DataFrame({'temporal_offset_seconds': sorted(offsets.astype(float))}), how='cross')
    deltas = expanded['temporal_offset_seconds'] * expanded['container_fps']
    expanded['frame_delta'] = s6_integer_series(deltas.round(), 'Rounded frame deltas')
    if expanded.duplicated(['finding_segment_id', 'frame_delta']).any():
        raise ValueError('Different offsets round to the same decoder frame; widen the configured offsets.')
    expanded['opencv_frame_index'] = expanded['target_opencv_frame_index'] + expanded['frame_delta']
    expanded['context_frame_number'] = expanded['opencv_frame_index'] - expanded['frame_index_offset']
    expanded['effective_temporal_offset_seconds'] = expanded['frame_delta'] / expanded['container_fps']
    expanded['is_target'] = expanded['temporal_offset_seconds'].eq(0)
    expanded['context_available'] = (expanded['opencv_frame_index'].ge(0) &
                                     expanded['opencv_frame_index'].lt(expanded['frame_count']) &
                                     expanded['context_frame_number'].ge(0))
    stats = expanded.groupby('finding_segment_id', as_index=False, sort=False, observed=True).agg(
        requested_context_count=('context_available', 'size'),
        available_context_count=('context_available', 'sum'),
        temporal_window_complete=('context_available', 'all'))
    result = expanded.loc[expanded['context_available']].merge(stats, on='finding_segment_id',
                                                               how='left', validate='many_to_one')
    if result.loc[result['is_target'], 'finding_segment_id'].nunique() != len(candidates):
        raise RuntimeError('Each eligible segment must retain exactly one verified target.')
    return s6_typed_table(result[S6_TEMPORAL_COLUMNS].sort_values(
        ['finding_segment_id', 'temporal_offset_seconds'], kind='stable'))


def build_temporal_window_report(temporal_manifest):
    return s6_typed_table(temporal_manifest[S6_WINDOW_COLUMNS].drop_duplicates().sort_values(
        'finding_segment_id', kind='stable'))


def add_temporal_context_metadata(temporal_manifest, labelled_manifest):
    """Enrich evidence without promoting an unlabelled frame to a normal finding.

    A verified frame may have several official image files. Keep every source
    annotation and candidate image reference, but select one image per temporal
    row. Targets use Sector 5's exact representative; other contexts prefer the
    same finding, then a stable annotation-ID order. Selected-image labels are
    separate from frame-wide labels. Only selected images enter the disk audit.
    """
    s6_require_columns(temporal_manifest, ['finding_segment_id', 'video_key',
        'context_frame_number', 'finding_class_normalized', 'is_target',
        'target_image_annotation_id', 'target_image_path', 'target_image_relpath'], 'Temporal manifest')
    s6_require_columns(labelled_manifest, ['annotation_id', 'video_key', 'frame_number',
        'finding_class_normalized', 'image_path', 'image_relpath'], 'Labelled manifest')
    keys = ['video_key', 'context_frame_number']
    labelled = labelled_manifest.rename(columns={'frame_number': 'context_frame_number'}).copy()
    if labelled[['video_key', 'context_frame_number', 'finding_class_normalized']].isna().any().any():
        raise ValueError('Verified annotation keys must be non-null.')
    if labelled['annotation_id'].isna().any() or not labelled['annotation_id'].is_unique:
        raise ValueError('Official annotation IDs must be unique and non-null.')
    if labelled[['image_path', 'image_relpath']].isna().any().any() or any(
           labelled[column].astype('string').str.strip().eq('').fillna(False).any()
           for column in ('image_path', 'image_relpath')):
        raise ValueError('Official image references must be non-null and non-empty.')
    labelled['context_frame_number'] = s6_integer_series(
        labelled['context_frame_number'], 'Verified context frame numbers', 0)
    for identity, counterpart in [('image_relpath', 'image_path'), ('image_path', 'image_relpath')]:
        if labelled.groupby(identity, observed=True)[counterpart].nunique(dropna=False).gt(1).any():
            raise ValueError('Conflicting official image path/relative-path mappings.')
    image_frames = labelled[['image_relpath'] + keys].drop_duplicates()
    if image_frames['image_relpath'].duplicated().any():
        raise ValueError('One official image reference belongs to different video/frame identities.')
    mapping_columns = [name for name in ['opencv_frame_index', 'frame_index_offset'] if name in labelled]
    if mapping_columns and labelled.groupby(keys, observed=True)[mapping_columns].nunique(dropna=False).gt(1).any().any():
        raise ValueError('Official annotations disagree on the verified video/frame mapping.')

    # Sorting representatives never removes an expert annotation or box.
    labelled = labelled.sort_values(keys + ['annotation_id', 'image_relpath'], kind='stable')
    labels_text = lambda values: '|'.join(sorted(values.astype(str).unique()))
    ids_json = lambda values: json.dumps(sorted(values.astype(str)), separators=(',', ':'))
    official = labelled.groupby(keys, as_index=False, observed=True, sort=True).agg(
        context_verified_labels=('finding_class_normalized', labels_text),
        context_verified_annotation_ids_json=('annotation_id', ids_json),
        context_verified_annotation_count=('annotation_id', 'size'))
    image_info = labelled.groupby(keys + ['image_path', 'image_relpath'], as_index=False,
                                  observed=True, sort=True).agg(
        context_image_annotation_ids_json=('annotation_id', ids_json),
        context_image_verified_labels=('finding_class_normalized', labels_text),
        context_image_annotation_count=('annotation_id', 'size'))
    image_info['_descriptor_json'] = [json.dumps({
        'image_path': row['image_path'], 'image_relpath': row['image_relpath'],
        'annotation_ids': json.loads(row['context_image_annotation_ids_json']),
        'finding_classes_normalized': row['context_image_verified_labels'].split('|'),
    }, sort_keys=True, separators=(',', ':')) for row in image_info.to_dict('records')]
    candidates = image_info.groupby(keys, as_index=False, observed=True, sort=True).agg(
        context_official_images_json=('_descriptor_json', lambda values: '[' + ','.join(values) + ']'),
        context_official_image_count=('image_relpath', 'size'))
    official = official.merge(candidates, on=keys, how='left', validate='one_to_one')
    official['context_is_verified'] = True

    image_columns = ['annotation_id', 'image_path', 'image_relpath']
    frame_choice = labelled.drop_duplicates(keys)[keys + image_columns].rename(columns={
        name: '_frame_' + name for name in image_columns})
    finding_keys = keys + ['finding_class_normalized']
    finding_choice = labelled.drop_duplicates(finding_keys)[finding_keys + image_columns].rename(columns={
        name: '_finding_' + name for name in image_columns})
    finding_choice['context_is_verified_same_finding'] = True
    target_choice = labelled[['annotation_id'] + finding_keys + ['image_path', 'image_relpath']].rename(columns={
        'annotation_id': 'target_image_annotation_id',
        **{name: '_target_' + name for name in finding_keys + ['image_path', 'image_relpath']},
    })
    generated = [c for c in temporal_manifest if c.startswith('context_is_verified') or c.startswith('context_official_')
                 or c.startswith('context_verified_') or c.startswith('context_image_')
                 or c in {'context_annotation_status', 'image_readable', 'image_validation_error',
                 'image_sha256', 'image_width', 'image_height'} or c in TEMPORAL_EXTRACTION_RESULT_COLUMNS[2:]]
    result = temporal_manifest.drop(columns=generated).merge(official, on=keys, how='left', validate='many_to_one')
    result = result.merge(frame_choice, on=keys, how='left', validate='many_to_one')
    result = result.merge(finding_choice, on=finding_keys, how='left', validate='many_to_one')
    result = result.merge(target_choice, on='target_image_annotation_id', how='left', validate='many_to_one')
    for column in ('context_is_verified', 'context_is_verified_same_finding'):
        result[column] = result[column].astype('boolean').fillna(False)
    targets = result['is_target'].astype('boolean')
    if targets.isna().any():
        raise ValueError('Target flags cannot be missing.')
    valid_target = result['context_is_verified_same_finding'].copy()
    for column in finding_keys:
        valid_target &= result[column].eq(result['_target_' + column]).fillna(False)
    for suffix in ['image_path', 'image_relpath']:
        valid_target &= result['target_' + suffix].eq(result['_target_' + suffix]).fillna(False)
    invalid_target = targets & ~valid_target
    if invalid_target.any():
        invalid = result.loc[invalid_target, ['finding_segment_id', 'video_key',
            'context_frame_number', 'target_image_annotation_id']].to_dict('records')
        raise RuntimeError(f'Temporal target could not be matched to its exact Sector 5 official image: {invalid}')
    for suffix in image_columns:
        selected = result['_finding_' + suffix].combine_first(result['_frame_' + suffix])
        target_column = 'target_image_annotation_id' if suffix == 'annotation_id' else '_target_' + suffix
        result['context_official_image_' + suffix if suffix == 'annotation_id' else 'context_official_' + suffix] = (
            selected.where(~targets, result[target_column]))
    selected_info = image_info.drop(columns='_descriptor_json').rename(columns={
        'image_path': 'context_official_image_path', 'image_relpath': 'context_official_image_relpath'})
    result = result.merge(selected_info, on=keys + ['context_official_image_path', 'context_official_image_relpath'],
                          how='left', validate='many_to_one')
    result['context_annotation_status'] = np.select(
        [targets.to_numpy(dtype=bool), result['context_is_verified_same_finding'].to_numpy(dtype=bool),
         result['context_is_verified'].to_numpy(dtype=bool)],
        ['verified_target', 'verified_same_finding', 'verified_other_finding'], default='unlabelled_context')
    result['context_image_path'] = result['context_official_image_path']
    result['context_image_relpath'] = result['context_official_image_relpath']
    result['context_image_source'] = np.where(result['context_official_image_path'].notna(),
                                             'official_labelled_image', 'video_extraction_required')
    result['context_image_selection_rule'] = np.select(
        [targets.to_numpy(dtype=bool), result['context_is_verified_same_finding'].to_numpy(dtype=bool),
         result['context_is_verified'].to_numpy(dtype=bool)],
        ['sector5_target_representative', 'same_finding_annotation', 'verified_frame_annotation'],
        default='video_extraction_required')
    result = result.drop(columns=[name for name in result if name.startswith(('_frame_', '_finding_', '_target_'))])
    return s6_typed_table(result)


def path_to_storage_relpath_or_missing(path, storage_root):
    """Lexical path conversion; no resolve/stat operations in a pure function."""
    if pd.isna(path):
        return pd.NA
    path, root = Path(str(path)), Path(str(storage_root))
    if not path.is_absolute() or not root.is_absolute() or '..' in path.parts:
        raise ValueError('Storage paths must be absolute and cannot contain parent traversal.')
    try:
        return path.relative_to(root).as_posix()
    except ValueError:
        raise ValueError(f'Image path lies outside declared storage_root: {path}') from None


def build_temporal_output_path(video_key, opencv_frame_index, dirs, config):
    key = str(video_key)
    if pd.isna(video_key) or re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]*', key) is None or key in {'.', '..'}:
        raise ValueError('video_key must be a safe single directory component.')
    index = int(s6_integer_series(pd.Series([opencv_frame_index]), 'Decoder index', 0).iloc[0])
    extension = str(config['temporal_context_image_format']).lower().lstrip('.')
    if extension not in {'jpg', 'jpeg', 'png'}:
        raise ValueError(f'Unsupported image format: {extension}')
    return Path(dirs['temporal_frames_dir']) / key / f'frame_{index:08d}.{extension}'


def build_temporal_extraction_requests(temporal_manifest, video_manifest, dirs, config):
    s6_require_columns(temporal_manifest, ['video_key', 'opencv_frame_index', 'frame_index_offset',
        'context_image_path', 'context_image_source'], 'Temporal extraction input')
    s6_require_columns(video_manifest, ['video_key', 'video_path', 'video_relpath'], 'Video manifest')
    if video_manifest['video_key'].isna().any() or not video_manifest['video_key'].is_unique:
        raise ValueError('Video manifest requires unique non-null video keys.')
    mask = temporal_manifest['context_image_source'].eq('video_extraction_required').fillna(False)
    if (mask & temporal_manifest['context_image_path'].notna()).any():
        raise ValueError('A frame requesting extraction cannot already have an image path.')
    requests = temporal_manifest.loc[mask, ['video_key', 'opencv_frame_index', 'frame_index_offset']].drop_duplicates()
    if requests.empty:
        return s6_empty_table(TEMPORAL_EXTRACTION_REQUEST_COLUMNS)
    requests = requests.copy()
    requests['opencv_frame_index'] = s6_integer_series(requests['opencv_frame_index'], 'Requested indices', 0)
    if requests.duplicated(['video_key', 'opencv_frame_index']).any():
        raise ValueError('One requested decoder frame has inconsistent verified offsets.')
    requests = requests.merge(video_manifest[['video_key', 'video_path', 'video_relpath']],
                              on='video_key', how='left', validate='many_to_one')
    if requests[['video_path', 'video_relpath']].isna().any().any():
        raise KeyError('Source videos required for extraction are missing from the inventory.')
    requests['output_path'] = [str(build_temporal_output_path(k, i, dirs, config))
                               for k, i in zip(requests['video_key'], requests['opencv_frame_index'])]
    requests['output_relpath'] = requests['output_path'].map(
        lambda p: path_to_storage_relpath_or_missing(p, config['storage_root']))
    if requests['output_path'].duplicated().any():
        raise RuntimeError('Different requested frames resolve to the same destination.')
    return s6_typed_table(requests[TEMPORAL_EXTRACTION_REQUEST_COLUMNS].sort_values(
        ['video_key', 'opencv_frame_index'], kind='stable'))


def validate_temporal_result_coverage(extraction_requests, extraction_results):
    keys = ['video_key', 'opencv_frame_index']
    s6_require_columns(extraction_results, TEMPORAL_EXTRACTION_RESULT_COLUMNS, 'Extraction results')
    if extraction_results.duplicated(keys).any():
        raise ValueError('Extraction results contain duplicate video/frame keys.')
    joined = extraction_requests[keys].merge(extraction_results[keys], on=keys,
                                            how='outer', validate='one_to_one', indicator=True)
    if not joined['_merge'].eq('both').all():
        raise RuntimeError('Extraction result keys do not exactly match the requested keys.')
    statuses = extraction_results['extraction_status']
    if not statuses.isin(['success', 'failed', 'pending']).all():
        raise ValueError('Unsupported extraction result status.')
    ready = statuses.eq('success')
    if extraction_results.loc[ready, ['extracted_image_path', 'extracted_image_sha256']].isna().any().any():
        raise ValueError('Successful extraction requires an image path and hash.')
    if extraction_results.loc[~ready, 'extracted_image_path'].notna().any():
        raise ValueError('Failed/pending extraction cannot expose an image path.')
    return True


def attach_temporal_extraction_results(temporal_manifest, extraction_results):
    s6_require_columns(temporal_manifest, ['video_key', 'opencv_frame_index',
        'context_official_image_path', 'context_official_image_relpath'], 'Temporal manifest')
    s6_require_columns(extraction_results, TEMPORAL_EXTRACTION_RESULT_COLUMNS, 'Extraction results')
    if extraction_results.duplicated(['video_key', 'opencv_frame_index']).any():
        raise ValueError('Extraction results contain duplicate keys.')
    columns = TEMPORAL_EXTRACTION_RESULT_COLUMNS[2:]
    result = temporal_manifest.drop(columns=[c for c in columns if c in temporal_manifest]).merge(
        extraction_results, on=['video_key', 'opencv_frame_index'], how='left', validate='many_to_one')
    successful = result['extraction_status'].eq('success').fillna(False)
    official = result['context_official_image_path'].notna()
    result['context_image_path'] = result['context_official_image_path'].combine_first(
        result['extracted_image_path'].where(successful))
    result['context_image_relpath'] = result['context_official_image_relpath'].combine_first(
        result['extracted_image_relpath'].where(successful))
    result['context_image_source'] = np.select([official.to_numpy(dtype=bool),
        successful.to_numpy(dtype=bool), result['extraction_status'].eq('pending').fillna(False).to_numpy(dtype=bool)],
        ['official_labelled_image', 'extracted_from_video', 'video_extraction_required'], default='missing')
    return s6_typed_table(result)


def attach_temporal_image_audit(temporal_manifest, image_audit):
    s6_require_columns(image_audit, S6_AUDIT_COLUMNS, 'Image audit')
    if image_audit['context_image_path'].duplicated().any():
        raise ValueError('Image audit requires one row per physical image path.')
    columns = ['image_readable', 'image_sha256', 'image_width', 'image_height', 'image_validation_error']
    result = temporal_manifest.drop(columns=[c for c in columns if c in temporal_manifest]).merge(
        image_audit[['context_image_path'] + columns], on='context_image_path', how='left', validate='many_to_one')
    result['image_readable'] = result['image_readable'].astype('boolean').fillna(False)
    result.loc[result['context_image_path'].isna(), 'image_validation_error'] = 'No context image is available.'
    return s6_typed_table(result)


def build_temporal_extraction_failures(temporal_manifest, config):
    mask = ~temporal_manifest['image_readable']
    if not config['temporal_context_extract_frames']:
        # Pending unlabelled requests are an intentional planning outcome.
        mask &= temporal_manifest['context_official_image_path'].notna()
    return s6_typed_table(temporal_manifest.loc[mask].reindex(columns=TEMPORAL_EXTRACTION_FAILURE_COLUMNS))


def build_temporal_status_report(temporal_manifest):
    keys = ['split', 'context_annotation_status', 'context_image_source', 'image_readable']
    return s6_typed_table(temporal_manifest.groupby(keys, dropna=False, observed=True,
        as_index=False, sort=True).size().rename(columns={'size': 'row_count'}))


def build_sector6_summary(tables):
    temporal, requests, results = (tables[n] for n in
        ['temporal_manifest', 'temporal_extraction_requests', 'extracted_temporal_frames'])
    failures, windows = tables['temporal_extraction_failures'], tables['temporal_window_report']
    pending = int(results['extraction_status'].eq('pending').sum())
    return {
        'schema_version': 1, 'phase': 2, 'sector': 6,
        'table_row_counts': {name: len(table) for name, table in tables.items()},
        'temporal_reference_rows': len(temporal), 'eligible_segment_windows': len(windows),
        'unique_context_frames': len(temporal[['video_key', 'opencv_frame_index']].drop_duplicates()),
        'unique_extraction_requests': len(requests), 'pending_extraction_requests': pending,
        'failed_extraction_requests': int(results['extraction_status'].eq('failed').sum()),
        'missing_or_unreadable_context_rows': int((~temporal['image_readable']).sum()),
        'failure_reference_rows': len(failures),
        'incomplete_temporal_windows': int((~windows['temporal_window_complete']).sum()),
        'official_image_reference_rows': int(temporal['context_image_source'].eq('official_labelled_image').sum()),
        'extracted_image_reference_rows': int(temporal['context_image_source'].eq('extracted_from_video').sum()),
        'unlabelled_context_reference_rows': int(temporal['context_annotation_status'].eq('unlabelled_context').sum()),
        'validation_passed': len(failures) == 0,
        'execution_outcome': 'failed' if len(failures) else 'planned' if pending else 'complete',
        'annotation_semantics': 'Unlabelled means no ground truth; it does not mean normal.',
        'temporal_semantics': 'Container-FPS offsets, not verified clinical boundaries or VFR timestamps.',
    }


### 6. Explicit extraction and physical-image I/O

NVDEC frames are decoded sequentially in display order using an independent ordinal cursor for each video. The scheduler runs at most 10 videos concurrently; each worker owns a separate PyNvVideoCodec `SimpleDecoder`, uses the GPU primary CUDA context, and gets an internally created CUDA stream. Batches hold at most four RGB device frames per worker plus the decoder's internal surfaces. Only requested images use DLPack to copy from GPU to host before encoding. No complete video or complete frame inventory is loaded into GPU memory. A source video is fingerprinted using SHA-256; cached images are accepted only when the source, verified mapping, NVDEC/PyTorch runtime, decoder identity, encoding settings and saved image hash agree. A readable file alone is insufficient.

Image bytes are encoded and verified before atomic installation. Missing videos, decode failures and write errors become result rows. The final audit inspects every referenced official or extracted image for physical readability and content hash. With extraction disabled, unresolved video frames remain `pending`; they are not recorded as completed extraction failures.

In [38]:
# -----------------------------------------------------------------------------------
# Sector 6 I/O: NVDEC decoding, provenance-aware reuse and physical audit.
# -----------------------------------------------------------------------------------

def io_s6_nvdec_runtime(config):
    """Require NVIDIA CUDA + PyNvVideoCodec explicitly; no CPU decode fallback."""
    if config['temporal_context_decoder'] != 'nvdec':
        raise ValueError("Sector 6 requires temporal_context_decoder='nvdec'.")
    try:
        torch = importlib.import_module('torch')
        nvc = importlib.import_module('PyNvVideoCodec')
    except (ImportError, OSError) as error:
        raise RuntimeError(
            'NVDEC requires PyNvVideoCodec and CUDA-enabled PyTorch. '
            'Select an NVIDIA GPU runtime in Colab and run the dependencies cell. '
            f'Import failed: {type(error).__name__}: {error}'
        ) from None
    if not torch.cuda.is_available():
        raise RuntimeError('NVDEC requires an NVIDIA GPU runtime; torch.cuda.is_available() is False.')
    gpu_id = config['temporal_context_gpu_id']
    if gpu_id >= torch.cuda.device_count():
        raise RuntimeError(f'Configured GPU {gpu_id} is unavailable in this runtime.')
    if not hasattr(nvc, 'SimpleDecoder') or not hasattr(getattr(nvc, 'OutputColorType', None), 'RGB'):
        raise RuntimeError('Install PyNvVideoCodec >= 2.0.4 for the required GPU decoder API.')
    version = getattr(nvc, '__version__', None)
    if version is None:
        version = importlib.metadata.version('PyNvVideoCodec')
    info = {
        'gpu_id': int(gpu_id), 'gpu_name': torch.cuda.get_device_name(gpu_id),
        'pynvvideocodec_version': str(version), 'torch_version': str(torch.__version__),
        'cuda_version': str(torch.version.cuda), 'decoder_api': 'SimpleDecoder',
        'decode_batch_size': int(config['temporal_context_decode_batch_size']),
    }
    return {'nvc': nvc, 'torch': torch, 'info': info}


def io_s6_nvdec_frame_to_bgr(frame, torch):
    """Copy only a selected RGB device surface before its decoder batch is released."""
    tensor = torch.from_dlpack(frame)
    try:
        rgb = tensor.cpu().numpy()
        if rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:
            raise ValueError('NVDEC RGB output must be an uint8 HWC image with three channels.')
        # cvtColor owns the resulting CPU image; no device tensor remains in the
        # writer or is accessed after the next get_batch_frames() call.
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    finally:
        del tensor


def get_temporal_image_write_parameters(output_path, config):
    extension = Path(output_path).suffix.lower()
    if extension in {'.jpg', '.jpeg'}:
        return [cv2.IMWRITE_JPEG_QUALITY, config['temporal_context_jpeg_quality']]
    if extension == '.png':
        return [cv2.IMWRITE_PNG_COMPRESSION, config['temporal_context_png_compression']]
    raise ValueError(f'Unsupported temporal image extension: {extension}')


def io_s6_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def is_readable_image_file(image_path):
    try:
        path = Path(image_path)
        if not path.is_file() or path.stat().st_size <= 0:
            return False
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        return image is not None and image.size > 0
    except (cv2.error, OSError, ValueError, TypeError):
        return False


def io_s6_atomic_bytes(path, payload, storage_root):
    """Install complete bytes in the same filesystem, without exposing partial files."""
    path, root = Path(path), Path(storage_root).resolve()
    if not path.is_absolute() or path.is_symlink():
        raise ValueError('Output must be an absolute non-symlink path.')
    path.parent.resolve().relative_to(root)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.parent.resolve().relative_to(root)
    descriptor, temporary = tempfile.mkstemp(prefix='.' + path.name + '-', dir=path.parent)
    try:
        with os.fdopen(descriptor, 'wb') as output:
            output.write(payload)
            output.flush()
            os.fsync(output.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def io_s6_cache_matches(output_path, identity, storage_root):
    path = Path(output_path)
    sidecar = path.with_name(path.name + '.provenance.json')
    try:
        if path.is_symlink() or sidecar.is_symlink():
            return None
        root = Path(storage_root).resolve()
        path.resolve().relative_to(root)
        sidecar.resolve().relative_to(root)
        if not sidecar.is_file() or not is_readable_image_file(path):
            return None
        saved = json.loads(sidecar.read_text(encoding='utf-8'))
        if saved.get('identity') != identity:
            return None
        digest = io_s6_file_sha256(path)
        return digest if saved.get('image_sha256') == digest else None
    except (OSError, ValueError, TypeError, json.JSONDecodeError):
        return None


def io_s6_write_frame(output_path, image, identity, config):
    """Encode and inspect bytes before atomic installation; sidecar commits last."""
    path = Path(output_path)
    ok, encoded = cv2.imencode(path.suffix, image, get_temporal_image_write_parameters(path, config))
    if not ok or encoded.size == 0:
        raise OSError('OpenCV could not encode the requested context image.')
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if decoded is None or decoded.shape[:2] != image.shape[:2]:
        raise OSError('Encoded context image could not be verified.')
    payload = encoded.tobytes()
    digest = hashlib.sha256(payload).hexdigest()
    io_s6_atomic_bytes(path, payload, config['storage_root'])
    if io_s6_file_sha256(path) != digest or not is_readable_image_file(path):
        raise OSError('Context image failed physical write verification.')
    record = json.dumps({'schema_version': 1, 'identity': identity, 'image_sha256': digest},
                        sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False)
    io_s6_atomic_bytes(path.with_name(path.name + '.provenance.json'), record.encode('utf-8'), config['storage_root'])
    return digest


def extract_temporal_frames_for_video(requests, config, decoder_signature, nvdec_runtime=None):
    """Decode once in order from ordinal 0; never perform per-frame random seek.

    Source/open/decode/write errors become one result row per requested frame.
    Runtime I/O failures therefore do not suppress the persisted failure report.
    """
    s6_require_columns(requests, TEMPORAL_EXTRACTION_REQUEST_COLUMNS, 'Extraction batch')
    if requests.empty:
        return s6_empty_table(TEMPORAL_EXTRACTION_RESULT_COLUMNS)
    if (requests['video_key'].nunique(dropna=False) != 1 or
            requests['video_path'].nunique(dropna=False) != 1 or
            requests['frame_index_offset'].nunique(dropna=False) != 1):
        raise ValueError('One extraction batch requires one video and one confirmed offset.')
    if requests.duplicated(['video_key', 'opencv_frame_index']).any():
        raise ValueError('Extraction batch has duplicate frame keys.')
    ordered = requests.copy()
    ordered['opencv_frame_index'] = s6_integer_series(ordered['opencv_frame_index'], 'Requested decoder indices', 0)
    ordered['frame_index_offset'] = s6_integer_series(ordered['frame_index_offset'], 'Verified offsets')
    ordered = ordered.sort_values('opencv_frame_index', kind='stable')
    dirs = {'temporal_frames_dir': Path(config['storage_root']) / config['temporal_context_image_dir']}
    for row in ordered.itertuples(index=False):
        expected = build_temporal_output_path(row.video_key, row.opencv_frame_index, dirs, config)
        if Path(row.output_path) != expected or row.output_relpath != path_to_storage_relpath_or_missing(expected, config['storage_root']):
            raise ValueError('Requested output path disagrees with the deterministic extraction plan.')
    source = Path(str(ordered['video_path'].iloc[0]))
    records = {int(row.opencv_frame_index): {
        'video_key': str(row.video_key), 'opencv_frame_index': int(row.opencv_frame_index),
        'extracted_image_path': pd.NA, 'extracted_image_relpath': pd.NA,
        'extraction_status': 'failed', 'extraction_error': 'Extraction did not complete.',
        'source_video_sha256': pd.NA, 'extracted_image_sha256': pd.NA,
        'decoder_backend': config['temporal_context_decoder'],
    } for row in ordered.itertuples(index=False)}
    runtime = nvdec_runtime if nvdec_runtime is not None else io_s6_nvdec_runtime(config)
    nvc, torch = runtime['nvc'], runtime['torch']
    decoder = None
    frames = None
    try:
        source.resolve().relative_to(Path(config['storage_root']).resolve())
        if not source.is_file() or source.is_symlink():
            raise OSError('Source video is missing or is a symlink.')
        before = source.stat()
        source_sha = io_s6_file_sha256(source)
        after_hash = source.stat()
        if (before.st_size, before.st_mtime_ns) != (after_hash.st_size, after_hash.st_mtime_ns):
            raise OSError('Source video changed during fingerprinting.')
        identities, pending = {}, []
        for row in ordered.itertuples(index=False):
            index = int(row.opencv_frame_index)
            record = records[index]
            record['source_video_sha256'] = source_sha
            identity = {
                'schema_version': 1, 'source_video_sha256': source_sha,
                'video_key': str(row.video_key), 'opencv_frame_index': index,
                'frame_index_offset': int(row.frame_index_offset),
                'decoder_backend': config['temporal_context_decoder'],
                'opencv_version': cv2.__version__, 'decoder_signature': decoder_signature,
                'nvdec_runtime': runtime['info'],
                'encoding_extension': Path(row.output_path).suffix.lower(),
                'encoding_parameters': get_temporal_image_write_parameters(row.output_path, config),
            }
            identities[index] = identity
            cached_sha = (io_s6_cache_matches(row.output_path, identity, config['storage_root'])
                          if config['temporal_context_reuse_existing'] else None)
            if cached_sha is not None:
                record.update(extraction_status='success', extraction_error=pd.NA,
                    extracted_image_path=str(row.output_path), extracted_image_relpath=str(row.output_relpath),
                    extracted_image_sha256=cached_sha)
            else:
                pending.append(row)
        if pending:
            # Each video worker owns its own decoder and internally created
            # CUDA stream. CUDA context 0 selects the GPU primary context.
            # Do not share decoders or hold frames across decoder batch calls.
            with torch.cuda.device(config['temporal_context_gpu_id']):
                decoder = nvc.SimpleDecoder(
                    str(source), gpu_id=config['temporal_context_gpu_id'],
                    cuda_context=0, cuda_stream=0, use_device_memory=True,
                    need_scanned_stream_metadata=False, decoder_cache_size=1,
                    output_color_type=nvc.OutputColorType.RGB,
                )
                next_index = 0
                pending_lookup = {int(row.opencv_frame_index): row for row in pending}
                batch_size = config['temporal_context_decode_batch_size']
                final_requested = max(pending_lookup)
                try:
                    while next_index <= final_requested:
                        # Consume sequential DISPLAY frames. The high-level
                        # decoder handles B-frame reordering and EOF draining.
                        # No reliance on approximate container len(decoder).
                        frames = decoder.get_batch_frames(min(batch_size, final_requested - next_index + 1))
                        if not frames:
                            raise OSError(f'NVDEC ended at ordinal {next_index}, before requested ordinal {final_requested}.')
                        for position in range(len(frames)):
                            index = next_index
                            next_index += 1
                            row = pending_lookup.get(index)
                            if row is None:
                                continue
                            record = records[index]
                            try:
                                image = io_s6_nvdec_frame_to_bgr(frames[position], torch)
                                digest = io_s6_write_frame(row.output_path, image, identities[index], config)
                                record.update(extraction_status='success', extraction_error=pd.NA,
                                    extracted_image_path=str(row.output_path), extracted_image_relpath=str(row.output_relpath),
                                    extracted_image_sha256=digest)
                            except Exception as error:
                                record['extraction_error'] = f'{type(error).__name__}: {error}'
                        # Drop GPU references before the next decoder batch.
                        frames = None
                except Exception as error:
                    detail = f'{type(error).__name__}: {error}'
                    for row in pending:
                        record = records[int(row.opencv_frame_index)]
                        if record['extraction_status'] != 'success':
                            record['extraction_error'] = detail
        final = source.stat()
        if (before.st_size, before.st_mtime_ns) != (final.st_size, final.st_mtime_ns):
            raise OSError('Source video changed during extraction; results were invalidated.')
    except Exception as error:
        # A missing/unopenable source invalidates cached rows too: never claim
        # a context image was verified against a source we could not inspect.
        for record in records.values():
            record.update(extraction_status='failed', extraction_error=f'{type(error).__name__}: {error}',
                          extracted_image_path=pd.NA, extracted_image_relpath=pd.NA, extracted_image_sha256=pd.NA)
    finally:
        frames = None
        # PyNvVideoCodec owns decoder/context resources; releasing the final
        # decoder reference destroys them without undocumented close APIs.
        decoder = None
    return s6_typed_table(pd.DataFrame(records.values())[TEMPORAL_EXTRACTION_RESULT_COLUMNS])


def run_temporal_extraction(extraction_requests, config, decoder_signature):
    """Run bounded independent video batches; thread completion never changes row order."""
    if extraction_requests.empty:
        return s6_empty_table(TEMPORAL_EXTRACTION_RESULT_COLUMNS)
    if not config['temporal_context_enabled'] or not config['temporal_context_extract_frames']:
        result = extraction_requests[['video_key', 'opencv_frame_index']].copy()
        for column in TEMPORAL_EXTRACTION_RESULT_COLUMNS[2:]:
            result[column] = pd.NA
        result['extraction_status'] = 'pending'
        result['decoder_backend'] = config['temporal_context_decoder']
        return s6_typed_table(result[TEMPORAL_EXTRACTION_RESULT_COLUMNS])
    batches = [group.copy() for _, group in extraction_requests.groupby('video_key', sort=True, observed=True)]
    runtime = io_s6_nvdec_runtime(config)
    results = []
    with ThreadPoolExecutor(max_workers=config['temporal_context_workers']) as executor:
        futures = {executor.submit(extract_temporal_frames_for_video, batch, config, decoder_signature, runtime): batch for batch in batches}
        for future in tqdm(as_completed(futures), total=len(futures), desc='Extracting temporal evidence'):
            results.append(future.result())
    result = s6_typed_table(pd.concat(results, ignore_index=True).sort_values(
        ['video_key', 'opencv_frame_index'], kind='stable'))
    validate_temporal_result_coverage(extraction_requests, result)
    return result


def io_audit_temporal_images(temporal_manifest):
    """Inspect each selected context image once; unselected candidates are references."""
    paths = temporal_manifest[['context_image_path', 'context_image_relpath']].dropna(
        subset=['context_image_path']).drop_duplicates().sort_values('context_image_path', kind='stable')
    if paths['context_image_path'].duplicated().any():
        raise ValueError('One physical image has conflicting portable references.')
    rows = []
    for row in tqdm(paths.itertuples(index=False), total=len(paths), desc='Checking context images'):
        result = {column: pd.NA for column in S6_AUDIT_COLUMNS}
        result.update(context_image_path=row.context_image_path, context_image_relpath=row.context_image_relpath,
                      image_readable=False)
        try:
            path = Path(row.context_image_path)
            if not path.is_file() or path.stat().st_size == 0:
                raise OSError('Context image is missing or empty.')
            image = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if image is None or image.size == 0:
                raise OSError('Context image cannot be decoded.')
            digest = io_s6_file_sha256(path)
            result.update(image_readable=True, image_sha256=digest,
                          image_width=image.shape[1], image_height=image.shape[0])
        except (cv2.error, OSError, ValueError) as error:
            result['image_validation_error'] = f'{type(error).__name__}: {error}'
        rows.append(result)
    return s6_typed_table(pd.DataFrame(rows, columns=S6_AUDIT_COLUMNS))


def io_s6_extend_lineage(lineage, extraction_requests, extraction_results, image_audit):
    """Bind consumed source files and official image bytes to the saved provenance."""
    result = copy.deepcopy(lineage)
    source_rows = extraction_results[['video_key', 'source_video_sha256']].dropna().drop_duplicates()
    source_paths = extraction_requests[['video_key', 'video_path']].drop_duplicates().set_index('video_key')['video_path']
    if source_rows['video_key'].duplicated().any():
        raise RuntimeError('One source video produced inconsistent content hashes.')
    for row in source_rows.itertuples(index=False):
        key = 'source_video/' + str(row.video_key)
        result['input_sha256'][key] = row.source_video_sha256
        result['input_descriptors'][key] = str(source_paths.loc[row.video_key])
    # Extracted images are output artifacts; their content hashes live in the
    # result and audit tables. Existing official images are consumed inputs.
    extracted_paths = set(extraction_results['extracted_image_path'].dropna().astype(str))
    for row in image_audit.loc[image_audit['image_readable']].itertuples(index=False):
        if str(row.context_image_path) not in extracted_paths:
            key = 'official_image/' + str(row.context_image_relpath)
            result['input_sha256'][key] = row.image_sha256
            result['input_descriptors'][key] = str(row.context_image_path)
    return result


In [39]:
def io_run_sector6(inputs, config, lineage, decoder_signature):
    """Compose the pure tables and I/O stages without depending on notebook globals."""
    labelled = inputs['sector5']['labelled_frame_manifest_with_segments']
    segments = inputs['sector5']['finding_segment_manifest']
    videos = inputs['sector3']['video_frame_manifest']
    dirs = {'temporal_frames_dir': str(Path(config['storage_root']) / config['temporal_context_image_dir'])}
    offsets = build_verified_video_offsets(labelled)
    temporal = build_temporal_manifest(segments, videos, config, offsets)
    windows = build_temporal_window_report(temporal)
    temporal = add_temporal_context_metadata(temporal, labelled)
    requests = build_temporal_extraction_requests(temporal, videos, dirs, config)
    results = run_temporal_extraction(requests, config, decoder_signature)
    validate_temporal_result_coverage(requests, results)
    temporal = attach_temporal_extraction_results(temporal, results)
    relative = temporal['context_image_path'].map(
        lambda path: path_to_storage_relpath_or_missing(path, config['storage_root'])).astype('string')
    if not relative.fillna('<missing>').eq(temporal['context_image_relpath'].fillna('<missing>')).all():
        raise RuntimeError('Live image paths disagree with their portable storage references.')
    audit = io_audit_temporal_images(temporal)
    extracted_hashes = results.loc[results['extraction_status'].eq('success'),
        ['extracted_image_path', 'extracted_image_sha256']].set_index('extracted_image_path')['extracted_image_sha256']
    expected_hashes = audit['context_image_path'].map(extracted_hashes)
    changed = expected_hashes.notna() & audit['image_sha256'].ne(expected_hashes).fillna(True)
    audit.loc[changed, 'image_readable'] = False
    audit.loc[changed, 'image_validation_error'] = 'Extracted image changed between extraction and physical audit.'
    temporal = attach_temporal_image_audit(temporal, audit)
    failures = build_temporal_extraction_failures(temporal, config)
    status = build_temporal_status_report(temporal)
    tables = {
        'temporal_manifest': temporal, 'temporal_extraction_requests': requests,
        'extracted_temporal_frames': results, 'temporal_window_report': windows,
        'temporal_extraction_failures': failures, 'temporal_image_audit': audit,
        'temporal_status_report': status,
    }
    updated_lineage = io_s6_extend_lineage(lineage, requests, results, audit)
    # These are executable dependencies too, even though the runtime identity
    # function only fingerprints this entrypoint without invoking it.
    build_sector6_summary(tables)
    return {'tables': tables, 'lineage': updated_lineage}


In [40]:
NVDEC_RUNTIME_INFO = None
if CONFIG["temporal_context_enabled"] and CONFIG["temporal_context_extract_frames"]:
    NVDEC_RUNTIME_INFO = io_s6_nvdec_runtime(CONFIG)["info"]
    LINEAGE["nvdec_runtime"] = NVDEC_RUNTIME_INFO
    display(pd.DataFrame([NVDEC_RUNTIME_INFO]))
    print(f"NVDEC ready: GPU {CONFIG['temporal_context_gpu_id']}, "
          f"{CONFIG['temporal_context_workers']} concurrent video workers, "
          f"{CONFIG['temporal_context_decode_batch_size']} frames per decode batch.")


,gpu_id,gpu_name,pynvvideocodec_version,torch_version,cuda_version,decoder_api,decode_batch_size
0,0,NVIDIA L4,2.2.3,2.11.0+cu130,13.0,SimpleDecoder,4


NVDEC ready: GPU 0, 10 concurrent video workers, 4 frames per decode batch.


### 7. Processing identity from the active functions

The identity includes active transformation/extraction functions, referenced constants and dependency versions. Execution timestamps and cell positions do not affect it. The image-cache decoder identity is separate, so a reporting-only change does not require video decoding again.

In [41]:
def _s5_core_code_value(code):
    """Describe executable code without filenames, source lines or debug positions."""
    fields = (
        "co_argcount", "co_posonlyargcount", "co_kwonlyargcount", "co_nlocals",
        "co_stacksize", "co_flags", "co_code", "co_consts", "co_names", "co_varnames",
        "co_name", "co_qualname", "co_cellvars", "co_freevars", "co_exceptiontable",
    )
    return {field: _s5_core_value(getattr(code, field)) for field in fields if hasattr(code, field)}


def _s5_core_value(value, active=None):
    """Canonicalize scalar/container code values and defaults without changing them."""
    if isinstance(value, CodeType):
        return {"code": _s5_core_code_value(value)}
    if value is Ellipsis:
        return {"ellipsis": True}
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return {"float": value.hex()} if math.isfinite(value) else {"float": repr(value)}
    if isinstance(value, complex):
        return {"complex": [_s5_core_value(value.real), _s5_core_value(value.imag)]}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if not isinstance(value, (dict, list, tuple, set, frozenset)):
        raise TypeError(f"Unsupported active core identity value: {type(value).__name__}")
    active = set() if active is None else active
    identity = id(value)
    if identity in active:
        raise ValueError("Cyclic containers cannot define a deterministic core identity.")
    nested = active | {identity}
    if isinstance(value, dict):
        pairs = [[_s5_core_value(key, nested), _s5_core_value(item, nested)] for key, item in value.items()]
        return {"dict": sorted(pairs, key=lambda pair: _s5_core_identity_json(pair[0]))}
    items = [_s5_core_value(item, nested) for item in value]
    if isinstance(value, (set, frozenset)):
        items = sorted(items, key=_s5_core_identity_json)
    return {type(value).__name__: items}


def _s5_core_identity_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def _s5_core_referenced_names(code):
    """Include globals referenced inside comprehensions and other nested code."""
    children = (item for item in code.co_consts if isinstance(item, CodeType))
    return set(code.co_names).union(*( _s5_core_referenced_names(child) for child in children))


def _s5_core_function_value(function):
    closure = tuple(cell.cell_contents for cell in (function.__closure__ or ()))
    return {
        "code": _s5_core_code_value(function.__code__),
        "defaults": _s5_core_value(function.__defaults__),
        "kwdefaults": _s5_core_value(function.__kwdefaults__),
        "closure": _s5_core_value(closure),
    }


def sector6_runtime_core_sha256(entrypoint, namespace, dependency_versions):
    """Fingerprint active core functions, referenced S3 constants and library versions.

    Only Python functions reachable through executable global references and
    referenced ``S5_*`` constants participate. Unrelated configuration, clocks and
    notebook state are ignored. Recursive function graphs terminate through a
    visited-name set; repeated cell executions keep the same identity.
    """
    if not isinstance(entrypoint, FunctionType) or not isinstance(namespace, dict):
        raise TypeError("Provide a Python entrypoint function and its explicit namespace dictionary.")
    if not isinstance(dependency_versions, dict):
        raise TypeError("Dependency versions must be an explicit dictionary.")
    pending = {"entrypoint": entrypoint}
    functions, constants = {}, {}
    while pending:
        name = min(pending)
        function = pending.pop(name)
        if name in functions:
            continue
        functions[name] = _s5_core_function_value(function)
        for reference in sorted(_s5_core_referenced_names(function.__code__)):
            value = namespace.get(reference)
            if isinstance(value, FunctionType) and reference not in functions:
                pending[reference] = value
            elif reference.isupper() and reference in namespace and isinstance(value, (str, bool, int, float, bytes, dict, list, tuple, set, frozenset, type(None))):
                constants[reference] = _s5_core_value(value)
    payload = {
        "schema_version": 1, "functions": functions, "constants": constants,
        "dependency_versions": _s5_core_value(dependency_versions),
    }
    return hashlib.sha256(_s5_core_identity_json(payload).encode("utf-8")).hexdigest()



In [42]:
DEPENDENCY_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in ("numpy", "pandas", "pyarrow", "opencv-python-headless", "PyNvVideoCodec", "torch", "tqdm")
}
DEPENDENCY_VERSIONS["python"] = platform.python_version()
PIPELINE_CORE_SHA256 = sector6_runtime_core_sha256(io_run_sector6, globals(), DEPENDENCY_VERSIONS)
INPUT_READER_SHA256 = sector6_runtime_core_sha256(io_load_sector6_context, globals(), DEPENDENCY_VERSIONS)
DECODER_SIGNATURE = sector6_runtime_core_sha256(extract_temporal_frames_for_video, globals(), DEPENDENCY_VERSIONS)
LINEAGE["core_source_sha256"] = _s3_semantic_digest({
    "pipeline": PIPELINE_CORE_SHA256, "input_reader": INPUT_READER_SHA256,
})
LINEAGE["dependency_versions"] = DEPENDENCY_VERSIONS
LINEAGE["decoder_signature"] = DECODER_SIGNATURE
print("Sector 6 active processing identity:", LINEAGE["core_source_sha256"])


Sector 6 active processing identity: d96ab506e82988962b2b84bf522e28a2f52a2f963908ba6a5a568756304798e5


### 8. Build, extract and audit the Sector 6 tables

Rerunning reloads checked inputs, rebuilds pure metadata, revalidates cached images, and avoids video decoding when valid cached outputs already exist. Annotation metadata and physical-image status remain separate.

In [43]:
SECTOR6_RUN = io_run_sector6(INPUTS, CONFIG, LINEAGE, DECODER_SIGNATURE)
SECTOR6_TABLES = SECTOR6_RUN["tables"]
LINEAGE = SECTOR6_RUN["lineage"]
TABLES = SECTOR6_TABLES

temporal_manifest = TABLES["temporal_manifest"]
temporal_extraction_requests = TABLES["temporal_extraction_requests"]
extracted_temporal_frames = TABLES["extracted_temporal_frames"]
temporal_window_report = TABLES["temporal_window_report"]
temporal_extraction_failures = TABLES["temporal_extraction_failures"]
temporal_image_audit = TABLES["temporal_image_audit"]
temporal_status_report = TABLES["temporal_status_report"]
SECTOR6_SUMMARY = build_sector6_summary(TABLES)
missing_context_count = SECTOR6_SUMMARY["missing_or_unreadable_context_rows"]

display(pd.DataFrame([{"table": name, "rows": len(table), "columns": len(table.columns)}
                      for name, table in TABLES.items()]))
display(temporal_status_report)
display(temporal_manifest.head())
display(extracted_temporal_frames["extraction_status"].value_counts(dropna=False))
if not temporal_extraction_failures.empty:
    display(temporal_extraction_failures.head(30))
print(json.dumps(SECTOR6_SUMMARY, indent=2))


Extracting temporal evidence:   0%|          | 0/40 [00:00<?, ?it/s]

Checking context images:   0%|          | 0/2921 [00:00<?, ?it/s]

,table,rows,columns
0,temporal_manifest,2979,53
1,temporal_extraction_requests,1403,7
2,extracted_temporal_frames,1403,9
3,temporal_window_report,600,7
4,temporal_extraction_failures,0,10
5,temporal_image_audit,2921,7
6,temporal_status_report,10,5


,split,context_annotation_status,context_image_source,image_readable,row_count
0,test,unlabelled_context,extracted_from_video,True,187
1,test,verified_same_finding,official_labelled_image,True,124
2,test,verified_target,official_labelled_image,True,79
3,train,unlabelled_context,extracted_from_video,True,1099
4,train,verified_other_finding,official_labelled_image,True,13
5,train,verified_same_finding,official_labelled_image,True,736
6,train,verified_target,official_labelled_image,True,466
7,validation,unlabelled_context,extracted_from_video,True,142
8,validation,verified_same_finding,official_labelled_image,True,78
9,validation,verified_target,official_labelled_image,True,55


,finding_segment_id,video_key,video_relpath,finding_class,finding_class_normalized,split,target_frame_number,target_opencv_frame_index,target_image_annotation_id,target_image_path,...,extraction_status,extraction_error,source_video_sha256,extracted_image_sha256,decoder_backend,image_readable,image_sha256,image_width,image_height,image_validation_error
0,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,data/raw/kvasir_capsule/osfstorage/labelled_vi...,Angiectasia,angiectasia,test,895,895,annotation_8fcec414c0fd2cf4_0,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,...,success,<NA>,cd013fe9262cc9e815fffdf471b46321ae2c9268aaefb9...,402d37e19582a276fd3cb9d8c2861813d72c950a366fb0...,nvdec,True,402d37e19582a276fd3cb9d8c2861813d72c950a366fb0...,336,336,<NA>
1,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,data/raw/kvasir_capsule/osfstorage/labelled_vi...,Angiectasia,angiectasia,test,895,895,annotation_8fcec414c0fd2cf4_0,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,...,success,<NA>,cd013fe9262cc9e815fffdf471b46321ae2c9268aaefb9...,19199d0011d2249a43a735063d1527d7d07697328f0472...,nvdec,True,19199d0011d2249a43a735063d1527d7d07697328f0472...,336,336,<NA>
2,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,data/raw/kvasir_capsule/osfstorage/labelled_vi...,Angiectasia,angiectasia,test,895,895,annotation_8fcec414c0fd2cf4_0,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,...,<NA>,<NA>,<NA>,<NA>,<NA>,True,8d2ba348591841a9bb1bc81cb20d3987888aa8fc10d5ce...,336,336,<NA>
3,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,data/raw/kvasir_capsule/osfstorage/labelled_vi...,Angiectasia,angiectasia,test,895,895,annotation_8fcec414c0fd2cf4_0,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,...,success,<NA>,cd013fe9262cc9e815fffdf471b46321ae2c9268aaefb9...,f720a6c291ea3e7bab1c9d502b8c60ddf70e86c8b0625b...,nvdec,True,f720a6c291ea3e7bab1c9d502b8c60ddf70e86c8b0625b...,336,336,<NA>
4,04a78ef00c5245e0__angiectasia__00001,04a78ef00c5245e0,data/raw/kvasir_capsule/osfstorage/labelled_vi...,Angiectasia,angiectasia,test,895,895,annotation_8fcec414c0fd2cf4_0,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,...,success,<NA>,cd013fe9262cc9e815fffdf471b46321ae2c9268aaefb9...,38244aa705eb614c12a10873866b4069f66d03a887c821...,nvdec,True,38244aa705eb614c12a10873866b4069f66d03a887c821...,336,336,<NA>


,count
extraction_status,
success,1403


{
  "schema_version": 1,
  "phase": 2,
  "sector": 6,
  "table_row_counts": {
    "temporal_manifest": 2979,
    "temporal_extraction_requests": 1403,
    "extracted_temporal_frames": 1403,
    "temporal_window_report": 600,
    "temporal_extraction_failures": 0,
    "temporal_image_audit": 2921,
    "temporal_status_report": 10
  },
  "temporal_reference_rows": 2979,
  "eligible_segment_windows": 600,
  "unique_context_frames": 2921,
  "unique_extraction_requests": 1403,
  "pending_extraction_requests": 0,
  "failed_extraction_requests": 0,
  "missing_or_unreadable_context_rows": 0,
  "failure_reference_rows": 0,
  "incomplete_temporal_windows": 13,
  "official_image_reference_rows": 1551,
  "extracted_image_reference_rows": 1428,
  "unlabelled_context_reference_rows": 1428,
  "validation_passed": true,
  "execution_outcome": "complete",
  "annotation_semantics": "Unlabelled means no ground truth; it does not mean normal.",
  "temporal_semantics": "Container-FPS offsets, not verified 

### 9. Persistent storage and Git publication helpers

Results use stable Parquet filenames. Reports use versioned CSV plus typed schema JSON; CSV missing values use the reserved `\N` sentinel. Summary/config snapshots use JSON. Current provenance is saved separately in `configs/sector6_lineage.json`, so changes to input bytes, code or dependencies remain visible even when configuration stays the same. A catalog commits last after typed readback and SHA checks, with rollback on installation errors. Unchanged saved bytes are reused, and configuration/report versions are created only when their content changes.

The catalog's `complete=True` means the artifact bundle was fully saved. Consumers must also require **summary `validation_passed=True` and `execution_outcome="complete"`** before using this bundle as completed image evidence. Failure/planning bundles are intentionally saved for review.

Git publishing starts from a freshly fetched remote branch in an isolated temporary checkout. Only cataloged Sector 6 files are staged. Pushes are ordinary, never forced; existing notebook checkouts and other sectors are preserved.

In [44]:
# -----------------------------------------------------------------------------------
# Sector 6 persistence: current catalog committed last; unchanged bytes reused.
# -----------------------------------------------------------------------------------

S6_STATE_PATH = 'configs/sector6_state.json'
S6_RECEIPT_PATH = 'configs/sector6_publication_receipt.json'
S6_LINEAGE_PATH = 'configs/sector6_lineage.json'
S6_RESULT_NAMES = frozenset({'temporal_manifest', 'temporal_extraction_requests', 'extracted_temporal_frames'})
S6_REPORT_NAMES = frozenset({'temporal_window_report', 'temporal_extraction_failures', 'temporal_image_audit', 'temporal_status_report'})
_p6_safe_path = _p3_safe_path
_p6_file_hash = _p3_file_hash


def _p6_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_p3_json_text(value), encoding='utf-8')
    if _p3_json_text(json.loads(path.read_text(encoding='utf-8'))) != _p3_json_text(value):
        raise RuntimeError(f'Sector 6 JSON readback failed: {path.name}')


def _p6_assert_no_secrets(value, path='configuration'):
    """Save credential references, never actual credential material."""
    if isinstance(value, dict):
        for key, item in value.items():
            if str(key).lower() in {'token', 'access_token', 'github_token', 'password',
                                   'api_key', 'authorization', 'secret', 'private_key', 'client_secret'} and item:
                raise ValueError(f'Credential material cannot be saved at {path}.{key}; use token_secret_name.')
            _p6_assert_no_secrets(item, f'{path}.{key}')
    elif isinstance(value, (list, tuple)):
        for item in value:
            _p6_assert_no_secrets(item, path)
    elif isinstance(value, str) and (
        re.search(r'https?://[^/\s]*:[^/@\s]+@', value)
        or re.search(r'(?:gh[pousr]_[A-Za-z0-9_]{20,}|github_pat_[A-Za-z0-9_]{20,})', value)
    ):
        raise ValueError(f'Credential material cannot be saved at {path}.')


def sector6_semantic_config_fingerprint(config):
    # Sector 4's projection already excludes clock metadata recursively.
    return _s3_semantic_digest(sector4_semantic_config(config))


def sector6_processing_signature(config, lineage):
    """Pure identity of the effective config, consumed input bytes and active code."""
    hashes, core_hash = lineage.get('input_sha256'), lineage.get('core_source_sha256')
    if not isinstance(hashes, dict) or not hashes or not isinstance(core_hash, str):
        raise ValueError('Sector 6 lineage needs input_sha256 and core_source_sha256.')
    return _p3_json_hash({'phase': 2, 'sector': 6, 'schema_version': 2,
                         'semantic_config_fingerprint': sector6_semantic_config_fingerprint(config),
                         'input_sha256': hashes, 'core_source_sha256': core_hash})


def _p6_lineage_payload(lineage):
    """Current provenance, excluding clock metadata while retaining input identities.

    Convert diagnostic DataFrames and other supported values before applying
    Sector 4's clock-only projection; hashes, descriptors, dependency versions,
    decoder settings and the upstream artifact chain keep their full meaning.
    """
    return sector4_semantic_config(json.loads(_p3_json_text(lineage)))


def _p6_root(config):
    declared = Path(config.get('sector6', {}).get('local_output_dir', 'outputs/phase2/sector6')).expanduser()
    return (declared if declared.is_absolute() else Path(config['storage_root']) / declared).resolve()


def _p6_validate_lineage(lineage):
    """Recheck every consumed upstream file immediately before publishing local results."""
    hashes, descriptors = lineage.get('input_sha256'), lineage.get('input_descriptors')
    if not isinstance(hashes, dict) or not hashes or not isinstance(descriptors, dict):
        raise ValueError('Sector 6 lineage needs input_sha256 and input_descriptors from checked input loading.')
    if set(hashes) != set(descriptors):
        raise ValueError('Sector 6 input file descriptors do not match its consumed input hashes.')
    for key, digest in hashes.items():
        if not isinstance(digest, str) or re.fullmatch(r'[0-9a-f]{64}', digest) is None:
            raise ValueError(f'Invalid consumed input SHA-256 in Sector 6 lineage: {key}')
        path = Path(descriptors[key]).expanduser()
        if not path.is_absolute() or not path.is_file() or path.is_symlink():
            raise RuntimeError(f'Consumed upstream artifact is unavailable or unsafe: {key}')
        if _p3_file_hash(path) != digest:
            raise RuntimeError(f'Consumed upstream artifact changed after loading: {key}; rerun input loading.')
    core_hash = lineage.get('core_source_sha256')
    if not isinstance(core_hash, str) or re.fullmatch(r'[0-9a-f]{64}', core_hash) is None:
        raise ValueError('Sector 6 lineage lacks the active transformation SHA-256; run the processing-signature cell.')


def _p6_read_state(root):
    """Validate the complete flat catalog, current config, schemas and report bytes."""
    root = Path(root).resolve()
    state_path = _p3_safe_path(root, S6_STATE_PATH)
    if not state_path.is_file():
        return None
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if state.get('schema_version') != 2 or state.get('complete') is not True:
        raise RuntimeError('Sector 6 catalog is incomplete or has an unsupported schema.')
    if state.get('content_id') != _p3_json_hash(_p3_catalog_meaning(state)):
        raise RuntimeError('Sector 6 catalog fingerprint is invalid.')
    records = {record['relative_path']: record for record in state['artifacts']}
    if len(records) != len(state['artifacts']) or {S6_STATE_PATH, S6_RECEIPT_PATH} & set(records):
        raise RuntimeError('Sector 6 catalog contains duplicate or operational artifact entries.')
    for relative, record in records.items():
        path = _p3_safe_path(root, relative)
        if not path.is_file() or path.stat().st_size != record['bytes'] or _p3_file_hash(path) != record['sha256']:
            raise RuntimeError(f'Sector 6 saved artifact checksum verification failed: {relative}')
    if state['config_snapshot'] not in records:
        raise RuntimeError('Sector 6 current configuration snapshot is absent from its catalog.')
    envelope = json.loads(_p3_safe_path(root, state['config_snapshot']).read_text(encoding='utf-8'))
    if envelope.get('semantic_config_fingerprint') != state['semantic_config_fingerprint'] or (
        sector6_semantic_config_fingerprint(envelope['effective_config']) != state['semantic_config_fingerprint']
    ):
        raise RuntimeError('Sector 6 saved effective configuration disagrees with its catalog.')
    if S6_LINEAGE_PATH not in records:
        raise RuntimeError('Sector 6 current lineage is absent from its catalog.')
    saved_lineage = json.loads(_p3_safe_path(root, S6_LINEAGE_PATH).read_text(encoding='utf-8'))
    if (saved_lineage.get('input_sha256') != state['input_sha256'] or
            saved_lineage.get('core_source_sha256') != state['core_source_sha256'] or
            not isinstance(saved_lineage.get('input_descriptors'), dict) or
            set(saved_lineage['input_descriptors']) != set(state['input_sha256'])):
        raise RuntimeError('Sector 6 saved current lineage disagrees with its catalog.')
    if _p3_json_hash(saved_lineage) != records[S6_LINEAGE_PATH]['sha256']:
        raise RuntimeError('Sector 6 current lineage is not canonically encoded.')
    for name, descriptor in state['tables'].items():
        if not {descriptor['relative_path'], descriptor['schema_path']}.issubset(records):
            raise RuntimeError(f'Sector 6 table/schema absent from catalog: {name}')
        schema = json.loads(_p3_safe_path(root, descriptor['schema_path']).read_text(encoding='utf-8'))
        if schema.get('artifact') != descriptor['relative_path'] or schema.get('fingerprint') != descriptor['fingerprint']:
            raise RuntimeError(f'Sector 6 saved table schema disagrees with catalog: {name}')
    summary = state['summary']
    if not {summary['summary_relative_path'], summary['report_relative_path']}.issubset(records):
        raise RuntimeError('Sector 6 summary/report is absent from its catalog.')
    saved_summary = json.loads(_p3_safe_path(root, summary['summary_relative_path']).read_text(encoding='utf-8'))
    if _p3_json_hash(saved_summary) != summary['fingerprint'] or (
        records[summary['report_relative_path']]['sha256'] != summary['report_fingerprint']
    ):
        raise RuntimeError('Sector 6 saved summary/report disagrees with its catalog.')
    for relative in state.get('obsolete_paths', []):
        if not _p3_relative(relative).startswith('results/') or relative in records:
            raise RuntimeError('Invalid Sector 6 obsolete-result deletion authorization.')
    return state


def io_load_cached_sector6(config, processing_signature):
    """Optional checked cache; no I/O writes or Git operations."""
    root = _p6_root(config)
    state = _p6_read_state(root)
    if state is None or state['processing_signature'] != processing_signature:
        return None
    return {'tables': {name: _p3_read_table(root, descriptor) for name, descriptor in state['tables'].items()},
            'state': state}


def _p6_summary(tables):
    """Pure summary supplied by the Sector 6 transformation cell."""
    return build_sector6_summary(tables)


def _p6_executive(summary):
    counts = summary.get('table_row_counts', {})
    return ('# Phase 2 — Sector 6: Temporal Context Images\n\n'
            'This sector enriches temporal context references using the accepted official labelled-frame '
            'manifest. Official images take priority. Unique context frames without an official image '
            'are planned for extraction from the checked source video, then joined back to every '
            'segment/context reference. Sector 4 physical-frame split assignments and all source annotations are retained.\n\n'
            f"The saved snapshot has **{counts.get('temporal_manifest', 0):,} temporal context references**, "
            f"**{counts.get('temporal_extraction_requests', 0):,} unique extraction requests**, and "
            f"**{counts.get('extracted_temporal_frames', 0):,} extraction result rows**. "
            f"The failure report contains **{counts.get('temporal_extraction_failures', 0):,} references**.\n\n"
            'An unlabelled context frame has no ground-truth finding annotation; it is not a verified '
            'normal frame. Other annotated findings retain their own labels. Source-video extraction '
            'does not create clinical ground truth or precise finding onset/offset.\n\n'
            'Image files remain in the declared persistent storage. GitHub receives the manifests, '
            'image audit, extraction errors, configuration snapshots, provenance and reports. '
            'Stable result filenames are verified and reused when content is unchanged; report and '
            'configuration versions are created only when their semantic content changes.\n')


def _p6_write_table(stage, name, frame, fingerprint, stamp):
    is_result = name in S6_RESULT_NAMES
    base = f'results/{name}' if is_result else f'reports/{name}_{stamp}'
    relative = base + ('.parquet' if is_result else '.csv')
    path = stage / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if is_result:
        frame.to_parquet(path, index=False, compression='zstd')
        restored = pd.read_parquet(path)
    else:
        serialized = frame.astype('string')
        if serialized.eq(P3_NULL).fillna(False).any().any():
            raise ValueError(f'Reserved CSV null sentinel occurs in {name}.')
        serialized.fillna(P3_NULL).to_csv(path, index=False)
        restored = _p3_restore_csv(path, _p3_schema(frame))
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), restored, check_exact=True, check_dtype=True)
    schema_relative = base + '_schema.json'
    _p6_write_json(stage / schema_relative, {**_p3_schema(frame), 'artifact': relative, 'fingerprint': fingerprint})
    return {'relative_path': relative, 'schema_path': schema_relative, 'fingerprint': fingerprint}


def _p6_install(stage, root, changed_paths, obsolete_paths):
    """Rollback on errors and replace the current state/catalog last."""
    if not root.exists():
        stage.rename(root)
        return
    backup = Path(tempfile.mkdtemp(prefix='.sector6-backup-', dir=root.parent))
    installed, moved = [], []
    sequence = [path for path in changed_paths if path != S6_STATE_PATH] + list(obsolete_paths) + [S6_STATE_PATH]
    try:
        for relative in dict.fromkeys(sequence):
            target = _p3_safe_path(root, relative)
            if target.exists():
                saved = backup / relative
                saved.parent.mkdir(parents=True, exist_ok=True)
                target.replace(saved)
                moved.append(relative)
            source = stage / relative
            if source.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                source.replace(target)
                installed.append(relative)
    except BaseException:
        for relative in reversed(installed):
            (root / relative).unlink(missing_ok=True)
        for relative in moved:
            target = root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            (backup / relative).replace(target)
        raise
    finally:
        shutil.rmtree(backup, ignore_errors=True)


def io_persist_sector6(tables, config, lineage):
    """Save/read back Sector 6 outputs; leave all saved files untouched when unchanged."""
    signature = sector6_processing_signature(config, lineage)
    if not isinstance(tables, dict) or not tables or any(not isinstance(frame, pd.DataFrame) for frame in tables.values()):
        raise TypeError('Sector 6 output tables must be a nonempty dictionary of DataFrames.')
    if set(tables) != S6_RESULT_NAMES | S6_REPORT_NAMES:
        raise ValueError('Pass all three Sector 6 result tables and all four report tables.')
    _p6_assert_no_secrets(config)
    _p6_assert_no_secrets(lineage)
    _p6_validate_lineage(lineage)
    root = _p6_root(config)
    previous = _p6_read_state(root)
    config_hash = sector6_semantic_config_fingerprint(config)
    provenance = {'input_sha256': copy.deepcopy(lineage['input_sha256']),
                  'core_source_sha256': lineage['core_source_sha256']}
    lineage_payload = _p6_lineage_payload(lineage)
    _p6_assert_no_secrets(lineage_payload, 'lineage')
    lineage_hash = _p3_json_hash(lineage_payload)
    fingerprints = {name: sector3_table_fingerprint(table) for name, table in tables.items()}
    summary = _p6_summary(tables)
    summary_hash = _p3_json_hash(summary)
    report_text = _p6_executive(summary)
    report_hash = hashlib.sha256(report_text.encode('utf-8')).hexdigest()
    same_tables = previous is not None and set(fingerprints) == set(previous['tables']) and all(
        previous['tables'][name]['fingerprint'] == value for name, value in fingerprints.items())
    if previous and same_tables and all(previous[key] == value for key, value in provenance.items()) and (
        previous['semantic_config_fingerprint'] == config_hash and previous['summary']['fingerprint'] == summary_hash
        and previous['processing_signature'] == signature and previous['summary']['report_fingerprint'] == report_hash
        and any(record['relative_path'] == S6_LINEAGE_PATH and record['sha256'] == lineage_hash
                for record in previous['artifacts'])
    ):
        for descriptor in previous['tables'].values():
            _p3_read_table(root, descriptor)
        return {'status': 'unchanged', 'root': str(root), 'local_root': str(root),
                'state': previous, 'artifacts': previous['artifacts'], 'changed_paths': []}
    root.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix='.sector6-stage-', dir=root.parent))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    try:
        catalog = {record['relative_path']: record for record in previous['artifacts']} if previous else {}
        descriptors, obsolete = {}, set(previous.get('obsolete_paths', [])) if previous else set()
        if catalog.get(S6_LINEAGE_PATH, {}).get('sha256') != lineage_hash:
            _p6_write_json(stage / S6_LINEAGE_PATH, lineage_payload)
        config_snapshot = previous['config_snapshot'] if previous else None
        if not previous or previous['semantic_config_fingerprint'] != config_hash:
            config_snapshot = f'configs/sector6_config_{stamp}.json'
            _p6_write_json(stage / config_snapshot, {'schema_version': 2, 'changed_at_utc': stamp,
                'semantic_config_fingerprint': config_hash, 'effective_config': config,
                'current_lineage_path': S6_LINEAGE_PATH, 'lineage': lineage})
        for name, frame in sorted(tables.items()):
            old = previous['tables'].get(name) if previous else None
            if old and old['fingerprint'] == fingerprints[name]:
                _p3_read_table(root, old)
                descriptors[name] = old
            else:
                descriptors[name] = _p6_write_table(stage, name, frame, fingerprints[name], stamp)
                if old and old['relative_path'].startswith('results/') and old['relative_path'] != descriptors[name]['relative_path']:
                    for relative in (old['relative_path'], old['schema_path']):
                        obsolete.add(relative)
                        catalog.pop(relative, None)
        previous_summary = previous['summary'] if previous else {}
        summary_path = previous_summary.get('summary_relative_path')
        if previous_summary.get('fingerprint') != summary_hash:
            summary_path = f'reports/summary_{stamp}.json'
            _p6_write_json(stage / summary_path, summary)
        report_path = previous_summary.get('report_relative_path')
        if catalog.get(report_path, {}).get('sha256') != report_hash:
            report_path = f'reports/executive_report_{stamp}.md'
            (stage / report_path).write_text(report_text, encoding='utf-8')
        summary_descriptor = {'fingerprint': summary_hash, 'summary_relative_path': summary_path,
                              'report_fingerprint': report_hash, 'report_relative_path': report_path}
        changed = []
        for path in sorted(stage.rglob('*')):
            if path.is_file():
                relative = path.relative_to(stage).as_posix()
                record = {'relative_path': relative, 'sha256': _p3_file_hash(path), 'bytes': path.stat().st_size}
                if catalog.get(relative) == record:
                    path.unlink()
                else:
                    catalog[relative] = record
                    changed.append(relative)
        obsolete.difference_update(catalog)
        state = {'schema_version': 2, 'complete': True, 'processing_signature': signature,
                 'semantic_config_fingerprint': config_hash, 'config_snapshot': config_snapshot,
                 **provenance, 'tables': descriptors, 'summary': summary_descriptor,
                 'artifacts': [catalog[path] for path in sorted(catalog)], 'obsolete_paths': sorted(obsolete)}
        state['content_id'] = _p3_json_hash(_p3_catalog_meaning(state))
        state['changed_at_utc'] = stamp
        _p6_write_json(stage / S6_STATE_PATH, state)
        changed.append(S6_STATE_PATH)
        _p6_validate_lineage(lineage)
        _p6_install(stage, root, changed, sorted(obsolete))
        _p6_read_state(root)
        for descriptor in state['tables'].values():
            _p3_read_table(root, descriptor)
        return {'status': 'updated', 'root': str(root), 'local_root': str(root),
                'state': state, 'artifacts': state['artifacts'], 'changed_paths': changed}
    finally:
        shutil.rmtree(stage, ignore_errors=True)


def _p6_read_github_token(secret_name):
    """Read a credential by reference; never print, save, or embed its value in a URL."""
    import os
    import re
    if not isinstance(secret_name, str) or not secret_name or re.search(r"\s", secret_name):
        raise ValueError("github.token_secret_name must be a nonempty name without whitespace.")
    token = os.environ.get(secret_name)
    if isinstance(token, str) and token.strip():
        return token.strip()
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError(
            f"GitHub secret {secret_name!r} is unavailable. Run from the Colab UI "
            "or provide an environment variable with this exact name."
        ) from None
    try:
        token = userdata.get(secret_name)
    except Exception as error:
        kind = type(error).__name__
        if kind == "SecretNotFoundError":
            detail = "Create this exact name in Colab Secrets or correct github.token_secret_name."
        elif kind == "NotebookAccessError":
            detail = "Enable Notebook access for this secret using the key icon in Colab."
        elif kind == "TimeoutException":
            detail = "Retry from an active Colab browser session."
        else:
            detail = "Check the exact name and Notebook access in Colab Secrets."
        raise RuntimeError(
            f"Cannot read GitHub secret {secret_name!r} ({kind}). {detail} "
            "The saved Sector 6 results can be reused."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f"GitHub secret {secret_name!r} is empty or invalid.")
    return token.strip()


def _p6_git_options(config):
    """Validate inherited repository settings and confine exports to Sector 6."""
    import re
    from pathlib import Path, PurePosixPath
    from urllib.parse import unquote, urlsplit
    github, sector = config.get("github", {}), config.get("sector6", {})
    if not isinstance(github, dict) or not isinstance(sector, dict):
        raise TypeError("github and sector6 configuration must be dictionaries.")
    def flag(name, default):
        value = sector.get(name, github.get(name, default))
        if not isinstance(value, bool):
            raise TypeError(f"{name} must be a Python/JSON boolean.")
        return value
    push, dry_run = flag("push", True), flag("dry_run", False)
    if not push and not dry_run:
        # Local-only runs need no repository URL, author, branch validation or token.
        return {
            "github": github, "push": False, "dry_run": False,
            "branch": github.get("branch", "main"),
            "export_dir": "outputs/phase2/sector6",
            "export_parts": PurePosixPath("outputs/phase2/sector6"),
        }
    allow_local = github.get("allow_local_test_remote", sector.get("allow_local_test_remote", False))
    if not isinstance(allow_local, bool):
        raise TypeError("allow_local_test_remote must be a Python/JSON boolean.")
    export_dir = sector.get("export_dir", github.get("repo_output_dir", "outputs/phase2/sector6"))
    if not isinstance(export_dir, str) or export_dir != "outputs/phase2/sector6":
        raise ValueError("Sector 6 exports must use outputs/phase2/sector6.")
    export_parts = PurePosixPath(export_dir)
    repo_url = github.get("repo_url")
    if not isinstance(repo_url, str) or not repo_url or repo_url != repo_url.strip():
        raise ValueError("github.repo_url must be configured.")
    parsed = urlsplit(repo_url)
    github_remote = (
        parsed.scheme == "https" and parsed.netloc == "github.com"
        and parsed.username is None and parsed.password is None
        and not parsed.query and not parsed.fragment
        and re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", parsed.path) is not None
    )
    if not github_remote:
        if not allow_local:
            raise ValueError("Only credential-free HTTPS github.com repository URLs are allowed.")
        if parsed.scheme == "file" and not parsed.netloc and not parsed.query and not parsed.fragment:
            remote_path = Path(unquote(parsed.path)).resolve()
        elif not parsed.scheme and not parsed.netloc:
            remote_path = Path(repo_url).resolve()
        else:
            raise ValueError("The explicitly enabled test remote must be a local filesystem repository.")
        if not remote_path.is_dir():
            raise ValueError("The local test repository does not exist.")
        repo_url = str(remote_path)
    branch = github.get("branch", "main")
    if not isinstance(branch, str) or not branch or branch.startswith("-") or any(c in branch for c in "\r\n\x00"):
        raise ValueError("Invalid Git branch name.")
    author_name = next((github[key] for key in (
        "git_username", "git_user_name", "git_author_name", "author_name", "github_username", "user_name"
    ) if github.get(key)), None)
    author_email = next((github[key] for key in (
        "git_email", "git_user_email", "git_author_email", "author_email", "user_email"
    ) if github.get(key)), None)
    if push and not dry_run:
        if not isinstance(author_name, str) or not author_name.strip() or not isinstance(author_email, str) or not author_email.strip():
            raise ValueError("Configure the inherited Git author name and email before publishing.")
        if any(c in author_name + author_email for c in "\r\n\x00"):
            raise ValueError("Git author name and email must be single-line values.")
    return {
        "github": github, "repo_url": repo_url, "github_remote": github_remote,
        "branch": branch, "push": push, "dry_run": dry_run,
        "export_dir": export_dir, "export_parts": export_parts,
        "author_name": author_name, "author_email": author_email,
    }


def _p6_git_session(options):
    """Return a private, temporary Git session; leave existing checkouts untouched."""
    from contextlib import contextmanager
    import os
    import subprocess
    import tempfile
    from pathlib import Path
    @contextmanager
    def session():
        secret_name = options["github"].get("token_secret_name", "GITHUB_TOKEN")
        token = _p6_read_github_token(secret_name) if options["github_remote"] and secret_name else None
        with tempfile.TemporaryDirectory(prefix="sector6-git-") as temporary:
            temporary_root = Path(temporary)
            checkout = temporary_root / "repository"
            checkout.mkdir()
            askpass = temporary_root / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
                "sys.stdout.write('x-access-token' if 'username' in prompt else os.environ.get('_P6_GITHUB_TOKEN', ''))\n",
                encoding="utf-8",
            )
            askpass.chmod(0o700)
            hooks = temporary_root / "hooks"
            hooks.mkdir()
            child_env = {
                key: value for key, value in os.environ.items()
                if not key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                and key not in {"GIT_CURL_VERBOSE", "SSH_ASKPASS"}
            }
            for variable in (
                "GIT_DIR", "GIT_WORK_TREE", "GIT_INDEX_FILE", "GIT_CONFIG", "GIT_CONFIG_COUNT", "GIT_COMMON_DIR",
                "GIT_OBJECT_DIRECTORY", "GIT_ALTERNATE_OBJECT_DIRECTORIES", "GIT_NAMESPACE", "_P6_GITHUB_TOKEN",
                "GIT_AUTHOR_NAME", "GIT_AUTHOR_EMAIL", "GIT_COMMITTER_NAME", "GIT_COMMITTER_EMAIL",
                "GIT_AUTHOR_DATE", "GIT_COMMITTER_DATE",
            ):
                child_env.pop(variable, None)
            child_env.update({
                "GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": str(askpass),
                "GIT_CONFIG_NOSYSTEM": "1", "GIT_CONFIG_GLOBAL": os.devnull,
                "GIT_LITERAL_PATHSPECS": "1",
            })
            if token:
                child_env["_P6_GITHUB_TOKEN"] = token
            def git(*args, allowed=(0,)):
                result = subprocess.run(
                    ["git", "-c", "credential.helper=", "-c", f"core.askPass={askpass}",
                     "-c", f"core.hooksPath={hooks}", *args],
                    cwd=checkout, env=child_env, text=True, stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE, check=False,
                )
                if result.returncode not in allowed:
                    # Do not expose remote-supplied output, which may contain credential material.
                    raise RuntimeError(f"Sector 6 Git {args[0]} failed (exit {result.returncode}).")
                return result
            git("check-ref-format", "--branch", options["branch"])
            git("init", "--quiet")
            git("remote", "add", "origin", options["repo_url"])
            yield checkout, git
    return session()


def io_fetch_sector6(config):
    """Fetch and report the latest remote tip without modifying the user's checkout."""
    options = _p6_git_options(config)
    if not options["push"] and not options["dry_run"]:
        return {"status": "not_requested", "branch": options["branch"]}
    with _p6_git_session(options) as (_, git):
        git("fetch", "--quiet", "origin", f"refs/heads/{options['branch']}")
        tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
        return {"status": "fetched", "branch": options["branch"], "remote_tip": tip}


def io_push_sector6(persisted, config):
    """Publish a catalog-verified Sector 6 snapshot with an ordinary non-force push.

    Only catalog artifacts, the Sector 6 state, and recorded obsolete result files
    are staged. The commit starts at the latest remote branch; races are retried.
    Existing checkouts, local edits and other pipeline sectors remain untouched.
    """
    import json
    import shutil
    from pathlib import Path, PurePosixPath
    options = _p6_git_options(config)
    branch, export_parts = options["branch"], options["export_parts"]
    outcome = {
        "status": "prepared", "branch": branch, "export_dir": options["export_dir"],
        "pushed": False, "dry_run": options["dry_run"],
    }
    if not options["push"] and not options["dry_run"]:
        return {**outcome, "status": "not_requested"}
    root_value = persisted.get("local_root", persisted.get("root"))
    if root_value is None:
        raise ValueError("The persisted Sector 6 root is missing.")
    root = Path(root_value).resolve()
    state = _p6_read_state(root)
    if state is None:
        raise RuntimeError("Sector 6 has no valid persisted snapshot to publish.")
    if persisted.get("state") != state:
        raise RuntimeError("Sector 6 changed after persistence; save it again before publishing.")
    artifacts = state.get("artifacts")
    if not isinstance(artifacts, list) or not artifacts:
        raise ValueError("Sector 6 state must contain a nonempty artifact catalog.")
    relative_paths, records = [], {}
    for artifact in artifacts:
        if not isinstance(artifact, dict):
            raise ValueError("Invalid Sector 6 artifact record.")
        relative = artifact.get("relative_path")
        source = _p6_safe_path(root, relative)
        if not source.is_file() or source.is_symlink():
            raise RuntimeError(f"Missing or unsafe Sector 6 artifact: {relative}")
        if _p6_file_hash(source) != artifact.get("sha256") or source.stat().st_size != artifact.get("bytes"):
            raise RuntimeError(f"Sector 6 artifact verification failed: {relative}")
        if relative in records:
            raise ValueError("Sector 6 artifact catalog has duplicate paths.")
        records[relative] = artifact
        relative_paths.append(relative)
    state_relative = "configs/sector6_state.json"
    state_source = _p6_safe_path(root, state_relative)
    if state_source.is_symlink() or json.loads(state_source.read_text(encoding="utf-8")) != state:
        raise RuntimeError("Sector 6 state changed during publisher verification.")
    if state_relative not in relative_paths:
        relative_paths.append(state_relative)
    obsolete_paths = state.get("obsolete_paths", [])
    if not isinstance(obsolete_paths, list) or any(not isinstance(path, str) for path in obsolete_paths):
        raise ValueError("Sector 6 obsolete_paths must be a list of recorded relative paths.")
    for relative in obsolete_paths:
        _p6_safe_path(root, relative)
        if not relative.startswith("results/") or relative in relative_paths:
            raise ValueError("Only obsolete recorded result files may be deleted from the export.")
    outcome["count"] = len(relative_paths)
    with _p6_git_session(options) as (checkout, git):
        # Each attempt starts from the newest remote branch, avoiding a hard reset/rebase.
        for attempt in range(1, 4):
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            remote_tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
            git("checkout", "--quiet", "--detach", remote_tip)
            def export_target(relative):
                relative_path = PurePosixPath(relative)
                if relative_path.is_absolute() or any(p in {".", "..", ".git"} for p in relative_path.parts) or "\\" in relative:
                    raise ValueError("Invalid export artifact path.")
                target = checkout.joinpath(*export_parts.parts, *relative_path.parts)
                current = checkout
                for part in (*export_parts.parts, *relative_path.parts):
                    current = current / part
                    if current.is_symlink():
                        raise RuntimeError("The Git export path contains a symlink.")
                return target
            staged_paths = []
            for relative in relative_paths:
                source, target = _p6_safe_path(root, relative), export_target(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, target)
                if relative in records:
                    record = records[relative]
                    if _p6_file_hash(target) != record["sha256"] or target.stat().st_size != record["bytes"]:
                        raise RuntimeError("A Sector 6 artifact changed while being copied.")
                if relative == state_relative and json.loads(target.read_text(encoding="utf-8")) != state:
                    raise RuntimeError("The exported Sector 6 state changed while being copied.")
                staged_paths.append(target.relative_to(checkout).as_posix())
            for relative in obsolete_paths:
                target = export_target(relative)
                git_relative = target.relative_to(checkout).as_posix()
                tracked = bool(git("ls-files", "--", git_relative).stdout.strip())
                if target.exists():
                    if not target.is_file():
                        raise RuntimeError("An obsolete Sector 6 result path is not a file.")
                    target.unlink()
                if tracked:
                    staged_paths.append(git_relative)
            git("add", "--", *sorted(set(staged_paths)))
            changed = git("diff", "--cached", "--name-only", "-z").stdout
            changed_paths = [path for path in changed.split("\x00") if path]
            if any(not path.startswith(options["export_dir"] + "/") for path in changed_paths):
                raise RuntimeError("The publication contains a change outside Sector 6.")
            outcome.update({"remote_tip": remote_tip, "changed_paths": changed_paths, "attempt": attempt})
            if _p6_read_state(root) != state:
                raise RuntimeError("Sector 6 changed during export; save it again before publishing.")
            if not changed_paths:
                return {**outcome, "status": "unchanged"}
            if options["dry_run"]:
                return {**outcome, "status": "dry_run"}
            git("-c", f"user.name={options['author_name']}", "-c", f"user.email={options['author_email']}",
                "commit", "--quiet", "-m", "Update Phase 2 Sector 6 temporal context image materialization")
            commit = git("rev-parse", "HEAD").stdout.strip()
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            latest_tip = git("rev-parse", "FETCH_HEAD").stdout.strip()
            if latest_tip != remote_tip:
                continue
            git("merge-base", "--is-ancestor", remote_tip, "HEAD")
            pushed = git("push", "--quiet", "origin", f"HEAD:refs/heads/{branch}", allowed=(0, 1, 128))
            if pushed.returncode == 0:
                return {**outcome, "status": "published", "pushed": True, "commit": commit}
            # Distinguish a concurrent remote update from permissions/network failures.
            git("fetch", "--quiet", "origin", f"refs/heads/{branch}")
            if git("rev-parse", "FETCH_HEAD").stdout.strip() == remote_tip:
                raise RuntimeError(
                    "Sector 6 Git push failed. Check repository access and token permissions; "
                    "the saved results remain available for retry."
                )
        raise RuntimeError("The remote branch kept changing; rerun the Sector 6 publication cell.")


### 10. Save and verify all outputs before the final validation gate

This step also saves failure reports when extraction is incomplete. It runs before any Git operation, so the mounted copy remains available if authentication or network publication fails.

In [45]:
PROCESSING_SIGNATURE = sector6_processing_signature(CONFIG, LINEAGE)
SECTOR6_PERSISTED = io_persist_sector6(TABLES, CONFIG, LINEAGE)
temporal_extraction_failure_path = Path(SECTOR6_PERSISTED["root"]) / SECTOR6_PERSISTED["state"]["tables"]["temporal_extraction_failures"]["relative_path"]
print("Sector 6 persistence:", SECTOR6_PERSISTED["status"])
print("Saved root:", SECTOR6_PERSISTED["root"])
display(pd.DataFrame(SECTOR6_PERSISTED["artifacts"]))


Sector 6 persistence: updated
Saved root: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/sector6


,relative_path,sha256,bytes
0,configs/sector6_config_20261005T235616462013Z....,794f54b1d630ecea4dcfc2e6e2a4597cb12a3ea84802da...,718762
1,configs/sector6_lineage.json,120d0896464c49e77de9ece30ec59765869a410ad56a29...,703604
2,reports/executive_report_20261005T235616462013...,3efb0fbb427a060e40509d224033125a1bc6003a17cb92...,1177
3,reports/summary_20261005T235616462013Z.json,ea7f6fe75e277fdf117ed3e53a27035a9dcc0a3a49b0d7...,1042
4,reports/temporal_extraction_failures_20261005T...,692e076cfa14ec5aa195e8fe657ec05ae59905a1620932...,196
5,reports/temporal_extraction_failures_20261005T...,e36f7f5accbb6baf6772e4a24b45d81f1eca7a851ba4da...,1288
6,reports/temporal_image_audit_20261005T23561646...,1e19745f314f95644e217f67321d2506d81dea040282be...,817426
7,reports/temporal_image_audit_20261005T23561646...,7e4b4d4109b405c24bf7f2b917ecc79bba9058b5a44ac5...,905
8,reports/temporal_status_report_20261005T235616...,88496cc767140cff56f343024965586fafc5bf28a77890...,662
9,reports/temporal_status_report_20261005T235616...,a8f72a6729b725139c1096265382a4b5ef7e44a569ae87...,725


### 11. Fetch the latest branch and publish the persisted bundle

`push=True` publishes the verified saved bundle, including any error reports. A missing Secret produces a specific actionable error without exposing credentials. Correct the Secret or publication config and rerun these cells; saved images/tables do not need to be discarded.

In [47]:
SECTOR6_FETCH = io_fetch_sector6(CONFIG)
print(SECTOR6_FETCH)


{'status': 'fetched', 'branch': 'main', 'remote_tip': '5bf2ac01214f354bc3ff5bd9b756c457abf34557'}


In [48]:
SECTOR6_PUBLICATION = io_push_sector6(SECTOR6_PERSISTED, CONFIG)
print(SECTOR6_PUBLICATION)


{'status': 'published', 'branch': 'main', 'export_dir': 'outputs/phase2/sector6', 'pushed': True, 'dry_run': False, 'count': 19, 'remote_tip': '5bf2ac01214f354bc3ff5bd9b756c457abf34557', 'changed_paths': ['outputs/phase2/sector6/configs/sector6_config_20261005T235616462013Z.json', 'outputs/phase2/sector6/configs/sector6_lineage.json', 'outputs/phase2/sector6/configs/sector6_state.json', 'outputs/phase2/sector6/reports/executive_report_20261005T235616462013Z.md', 'outputs/phase2/sector6/reports/summary_20261005T235616462013Z.json', 'outputs/phase2/sector6/reports/temporal_extraction_failures_20261005T235616462013Z.csv', 'outputs/phase2/sector6/reports/temporal_extraction_failures_20261005T235616462013Z_schema.json', 'outputs/phase2/sector6/reports/temporal_image_audit_20261005T235616462013Z.csv', 'outputs/phase2/sector6/reports/temporal_image_audit_20261005T235616462013Z_schema.json', 'outputs/phase2/sector6/reports/temporal_status_report_20261005T235616462013Z.csv', 'outputs/phase2/sec

### 12. Final validation gate

The failure report is already persistent at this point. Boundary-clipped windows are reported but are not extraction failures. Planning-only runs are explicitly marked as planned.

In [49]:
if not SECTOR6_SUMMARY["validation_passed"]:
    raise RuntimeError(
        f"Sector 6 has {SECTOR6_SUMMARY['failure_reference_rows']:,} missing/unreadable image references. "
        f"Review the saved failure report: {temporal_extraction_failure_path}"
    )
if not CONFIG["temporal_context_enabled"]:
    print("Sector 6 saved: temporal context is disabled.")
elif SECTOR6_SUMMARY["execution_outcome"] == "planned":
    print(f"Sector 6 plan saved: {SECTOR6_SUMMARY['pending_extraction_requests']:,} extraction requests remain pending.")
else:
    print("PASS: all available temporal context references have verified readable image files.")
print("Sector 6 outcome:", SECTOR6_SUMMARY["execution_outcome"])


PASS: all available temporal context references have verified readable image files.
Sector 6 outcome: complete
